<a href="https://colab.research.google.com/github/Vanilsondasilva/Vanilsondasilva/blob/main/base_analise_angiotomo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# Célula 1: Preparação do Ambiente
# =============================================================================

# Execute esta célula primeiro para instalar as dependências
print("🔧 Instalando dependências...")

# Instalar bibliotecas Python (silencioso)
!pip install dowhy econml pandas matplotlib seaborn scikit-learn openpyxl -qq

# Instalar dependências do sistema (muito silencioso)
!apt-get update -qq > /dev/null 2>&1
!apt-get install -qq graphviz graphviz-dev > /dev/null 2>&1

# Instalar bibliotecas de grafo (silencioso)
!pip install pygraphviz pydot -qq

print("📦 Dependências instaladas!")

# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("✅ Ambiente preparado com sucesso!")

🔧 Instalando dependências...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.2/399.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.0/193.0 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 25.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
📦 Dependências instaladas!
✅ Ambiente preparado com sucesso!


In [2]:
# =============================================================================
# Célula 2: Upload do Arquivo
# =============================================================================

# Execute esta célula para fazer upload do arquivo
uploaded = files.upload()

# Pegar o nome do arquivo que você fez upload
file_name = list(uploaded.keys())[0]
print(f"📁 Arquivo carregado: {file_name}")

Saving base_1_angio.xlsx to base_1_angio.xlsx
📁 Arquivo carregado: base_1_angio.xlsx


In [3]:
# =============================================================================
# Célula 3: Carregamento Inicial dos Dados
# =============================================================================

# Carregar os dados
if file_name.endswith('.xlsx'):
    df = pd.read_excel(file_name)
else:
    df = pd.read_csv(file_name)

print(f"📊 Dados carregados: {df.shape[0]:,} registros, {df.shape[1]} colunas")
print(f"📅 Colunas disponíveis: {list(df.columns)}")

# Visualizar primeiras linhas
df.head()

📊 Dados carregados: 593,597 registros, 19 colunas
📅 Colunas disponíveis: ['ID_PESSOA', 'DATA_NASCIMENTO_USU', 'DATA_ANGIOTOMO', 'AGRUP_ASSISTENCIAL_G', 'GRUPO_ESTATISTICO_G', 'DATA_ATENDIMENTO_FATO_PRO', 'NK_CODIGO_SERVICO', 'SERVICO', 'ESPECIALIDADE', 'DESCRICAO_CID', 'DESCRICAO_TIPO_INTERNAMENTO', 'CHV_INTERNAMENTO', 'DATA_INTERNACAO', 'DATA_ALTA', 'MOMENTO', 'DIFERENÇA EM DIA', 'MOMENTO_MES', 'FAIXA_TEMPORAL', 'ANGIOTOMO']


,ID_PESSOA,DATA_NASCIMENTO_USU,DATA_ANGIOTOMO,AGRUP_ASSISTENCIAL_G,GRUPO_ESTATISTICO_G,DATA_ATENDIMENTO_FATO_PRO,NK_CODIGO_SERVICO,SERVICO,ESPECIALIDADE,DESCRICAO_CID,DESCRICAO_TIPO_INTERNAMENTO,CHV_INTERNAMENTO,DATA_INTERNACAO,DATA_ALTA,MOMENTO,DIFERENÇA EM DIA,MOMENTO_MES,FAIXA_TEMPORAL,ANGIOTOMO
0,25702,1963-09-07,2025-03-07,3-EXAME,SADT - MEDICINA LABORATORIAL,2025-01-07,40301150,ACIDO URICO - PESQUISA E/OU DOSAGEM,NAO INFORMADO,ND,Nao Informado,-,-,-,ANTES,-59,-2,1 a 3 meses antes,Fora de Internamento
1,25702,1963-09-07,2025-03-07,3-EXAME,SADT - MEDICINA LABORATORIAL,2025-01-28,40301150,ACIDO URICO - PESQUISA E/OU DOSAGEM,NAO INFORMADO,ND,Nao Informado,-,-,-,ANTES,-38,-1,1 a 3 meses antes,Fora de Internamento
2,25702,1963-09-07,2025-03-07,6-OUTROS/ATEN._AMBULATORIAIS,HOS - TAXA DE EQUIPAMENTO /,2023-06-23,60024275,ALUGUEL/TAXA DE APARELHO / EQUIPAMENTO PARA CO...,NAO INFORMADO,ND,Nao Informado,-,-,-,ANTES,-623,-20,>12 meses antes,Fora de Internamento
3,25702,1963-09-07,2025-03-07,6-OUTROS/ATEN._AMBULATORIAIS,HOS - TAXA DE EQUIPAMENTO /,2023-06-23,60024330,ALUGUEL/TAXA DE APARELHO / EQUIPAMENTO PARA EN...,NAO INFORMADO,ND,Nao Informado,-,-,-,ANTES,-623,-20,>12 meses antes,Fora de Internamento
4,25702,1963-09-07,2025-03-07,3-EXAME,EX - IMA - IMAGEM - RESSONANCI,2025-03-04,41101537,ANGIO-RM ARTERIAL DE CRANIO,RADIOLOGIA,ND,Nao Informado,-,-,-,ANTES,-3,0,Até 30 dias antes,Fora de Internamento


In [15]:
# =============================================================================
# Célula 4 (corrigida): Jornada completa dos pacientes que tiveram INFARTO
# (Versão sem as tabelas longas de exemplo)
# =============================================================================
print("🔍 Iniciando análise CORRIGIDA: jornada completa dos pacientes com infarto...")

# 1) identificar IDs que em algum momento tiveram INFARTO (qualquer linha)
mask_infarto_any = df['DESCRICAO_CID'].str.contains('INFARTO AGUDO', case=False, na=False)
ids_infarto = df.loc[mask_infarto_any, 'ID_PESSOA'].unique()
print(f"🫀 IDs únicos com registro de INFARTO: {len(ids_infarto)}")

# 2) obter todas as linhas desses pacientes (toda a jornada)
df_jornada_infarto = df[df['ID_PESSOA'].isin(ids_infarto)].copy()
print(f"📂 Registros desses pacientes (toda jornada): {df_jornada_infarto.shape[0]:,}")

# 3) garantir colunas de interesse como texto (evita problemas de NaN/type)
for c in ['AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'MOMENTO', 'DESCRICAO_CID']:
    df_jornada_infarto[c] = df_jornada_infarto[c].astype(str)

# 4) definir masks para emergência e para consultas cardiologia eletiva
mask_emerg = df_jornada_infarto['AGRUP_ASSISTENCIAL_G'].str.contains('EMERGENCIA', case=False, na=False)
# Eletiva pode aparecer como '1-ELETIVA' etc. Usamos ELETIVA e cardiologia na especialidade
mask_eletiva = df_jornada_infarto['AGRUP_ASSISTENCIAL_G'].str.contains('ELETIVA', case=False, na=False)
mask_cardio = df_jornada_infarto['ESPECIALIDADE'].str.contains('CARDIO', case=False, na=False)

# 5) separar antes/depois pelo campo MOMENTO (assume que já existe)
antes = df_jornada_infarto[df_jornada_infarto['MOMENTO'].str.upper().str.strip() == 'ANTES']
depois = df_jornada_infarto[df_jornada_infarto['MOMENTO'].str.upper().str.strip() == 'DEPOIS']

# 6) métricas desejadas:
# - atendimentos de emergência (total e pacientes únicos) antes/depois
emerg_antes_total = antes[mask_emerg.loc[antes.index]].shape[0]
emerg_depois_total = depois[mask_emerg.loc[depois.index]].shape[0]

emerg_antes_pacientes = antes[mask_emerg.loc[antes.index]]['ID_PESSOA'].nunique()
emerg_depois_pacientes = depois[mask_emerg.loc[depois.index]]['ID_PESSOA'].nunique()

# - consultas eletivas com cardiologista (total e pacientes únicos)
cardio_elet_antes_total = antes[mask_eletiva.loc[antes.index] & mask_cardio.loc[antes.index]].shape[0]
cardio_elet_depois_total = depois[mask_eletiva.loc[depois.index] & mask_cardio.loc[depois.index]].shape[0]

cardio_elet_antes_pacientes = antes[mask_eletiva.loc[antes.index] & mask_cardio.loc[antes.index]]['ID_PESSOA'].nunique()
cardio_elet_depois_pacientes = depois[mask_eletiva.loc[depois.index] & mask_cardio.loc[depois.index]]['ID_PESSOA'].nunique()

# 7) montar resumo
resumo = pd.DataFrame({
    'Período': ['Antes', 'Depois'],
    'Pacientes únicos (nesse período)': [antes['ID_PESSOA'].nunique(), depois['ID_PESSOA'].nunique()],
    'Atendimentos emergência (total)': [emerg_antes_total, emerg_depois_total],
    'Pacientes com emergência': [emerg_antes_pacientes, emerg_depois_pacientes],
    'Consultas cardio eletiva (total)': [cardio_elet_antes_total, cardio_elet_depois_total],
    'Pacientes com cardio eletiva': [cardio_elet_antes_pacientes, cardio_elet_depois_pacientes]
})
print("📊 Resumo corrigido:")
display(resumo)

# 8) Top especialidades (antes / depois) - contagem completa (não apenas top10)
esp_antes = antes['ESPECIALIDADE'].value_counts().reset_index()
esp_antes.columns = ['Especialidade', 'Qtd_Antes']

esp_depois = depois['ESPECIALIDADE'].value_counts().reset_index()
esp_depois.columns = ['Especialidade', 'Qtd_Depois']

esp_comparativo = pd.merge(esp_antes, esp_depois, on='Especialidade', how='outer').fillna(0).sort_values(by=['Qtd_Antes','Qtd_Depois'], ascending=False)
print("📋 Especialidades (antes/depois) — tabela completa (ordenada):")
display(esp_comparativo.head(50)) # Mantive este .head(50), mas podes remover ou alterar se também estiver muito grande.

# 9) Exibir exemplos de registros onde há EMERGÊNCIA (para inspeção manual)
# print("🔎 Exemplos de atendimentos de EMERGÊNCIA (antes):")
# display(antes[mask_emerg.loc[antes.index]].head(20)) # <-- MODIFICADO: Comentado para não exibir a tabela grande

# print("🔎 Exemplos de atendimentos de EMERGÊNCIA (depois):")
# display(depois[mask_emerg.loc[depois.index]].head(20)) # <-- MODIFICADO: Comentado para não exibir a tabela grande

# 10) Linha do tempo por paciente: contagem de eventos por MOMENTO e tipo (Emergency/Cardio/Elective/other)
def tipo_atendimento(row):
    if pd.isna(row['AGRUP_ASSISTENCIAL_G']): return 'OUTRO'
    a = str(row['AGRUP_ASSISTENCIAL_G']).upper()
    e = str(row['ESPECIALIDADE']).upper()
    if 'EMERGENCIA' in a: return 'EMERGENCIA'
    if 'ELETIVA' in a and 'CARDIO' in e: return 'CARDIO_ELETIVA'
    if 'ELETIVA' in a: return 'ELETIVA_OUTRA'
    return 'OUTRO'

df_jornada_infarto['TIPO_ATEND'] = df_jornada_infarto.apply(tipo_atendimento, axis=1)

timeline = df_jornada_infarto.groupby(['ID_PESSOA','MOMENTO','TIPO_ATEND']).size().unstack(fill_value=0).reset_index()
print("🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):")
display(timeline.head(50)) # <-- MODIFICADO: Comentado para não exibir a tabela grande

print("✅ Análise corrigida executada. Agora apenas os resumos principais serão exibidos.")

🔍 Iniciando análise CORRIGIDA: jornada completa dos pacientes com infarto...
🫀 IDs únicos com registro de INFARTO: 2
📂 Registros desses pacientes (toda jornada): 1,421
📊 Resumo corrigido:


,Período,Pacientes únicos (nesse período),Atendimentos emergência (total),Pacientes com emergência,Consultas cardio eletiva (total),Pacientes com cardio eletiva
0,Antes,2,10,2,13,1
1,Depois,2,1,1,6,1


📋 Especialidades (antes/depois) — tabela completa (ordenada):


,Especialidade,Qtd_Antes,Qtd_Depois
20,NAO INFORMADO,731.0,260.0
4,CARDIOLOGIA,89.0,38.0
27,PSICOLOGIA,48.0,14.0
12,FISIOTERAPIA,22.0,14.0
1,ANESTESIOLOGIA,15.0,9.0
29,RADIOLOGIA,15.0,1.0
15,GINECOLOGIA E OBSTETRICIA,11.0,0.0
9,CLINICA MEDICA,10.0,4.0
22,OFTALMOLOGIA,10.0,0.0
23,OTORRINOLARINGOLOGIA,9.0,1.0


🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):


TIPO_ATEND,ID_PESSOA,MOMENTO,CARDIO_ELETIVA,ELETIVA_OUTRA,EMERGENCIA,OUTRO
0,333908,ANTES,0,30,7,590
1,333908,DEPOIS,0,7,1,123
2,333908,MESMA DATA,0,0,1,49
3,356645,ANTES,13,7,3,364
4,356645,DEPOIS,6,2,0,213
5,356645,MESMA DATA,0,0,0,5


✅ Análise corrigida executada. Agora apenas os resumos principais serão exibidos.


In [29]:
# =============================================================================
# Célula 5: Análise 3 - Busca por procedimentos-chave (Angio/Cateter/Stent)
# =============================================================================
print("Iniciando Análise 3: Busca por procedimentos-chave (Angio/Cateter/Stent)...")

# 1. Palavras-chave dos procedimentos
keywords_procedimentos = [
    'ANGIOTOMOGRAFIA CORONARIANA',
    'CATETERISMO CARDIACO',
    'IMPLANTE DE STENT CORONARIO'
]

# 2. Padrão regex com OU
pattern = '|'.join(keywords_procedimentos)

# 3. Filtrar registros com os procedimentos
df_jornada_infarto['SERVICO'] = df_jornada_infarto['SERVICO'].astype(str)
mask = df_jornada_infarto['SERVICO'].str.contains(pattern, case=False, na=False)
df_proc = df_jornada_infarto[mask].copy()

# 4. REMOVER DUPLICATAS: mesmo paciente + mesma data + mesmo serviço = 1 evento
df_proc_unicos = df_proc.drop_duplicates(
    subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'SERVICO']
)

# 5. Selecionar apenas colunas relevantes (sem ESPECIALIDADE)
colunas_display = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'MOMENTO', 'SERVICO']

# 6. Resultados
if df_proc_unicos.empty:
    print("\nRESULTADO: Nenhum procedimento-chave encontrado.")
else:
    print(f"\nSUCESSO! Encontrados {df_proc_unicos.shape[0]} procedimentos-chave únicos.\n")

    print("-- Detalhe dos Procedimentos (1 por data/serviço) --")
    display(df_proc_unicos[colunas_display]
            .sort_values(by=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO']))

    print("\n-- Resumo por Paciente e Momento --")
    resumo = (df_proc_unicos
              .groupby(['ID_PESSOA', 'MOMENTO', 'SERVICO'])
              .size()
              .reset_index(name='Qtd_Eventos'))
    display(resumo)

print("\nAnálise 3 concluída.")

Iniciando Análise 3: Busca por procedimentos-chave (Angio/Cateter/Stent)...

SUCESSO! Encontrados 7 procedimentos-chave únicos.

-- Detalhe dos Procedimentos (1 por data/serviço) --


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,MOMENTO,SERVICO
254385,333908,2022-09-27,ANTES,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
254950,333908,2024-11-19,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
254387,333908,2024-11-20,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
274838,356645,2022-04-11,ANTES,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
275251,356645,2024-08-23,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
274840,356645,2024-09-24,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
275085,356645,2024-11-05,DEPOIS,IMPLANTE DE STENT CORONARIO COM OU SEM ANGIOPL...



-- Resumo por Paciente e Momento --


,ID_PESSOA,MOMENTO,SERVICO,Qtd_Eventos
0,333908,ANTES,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
1,333908,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
2,333908,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
3,356645,ANTES,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
4,356645,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
5,356645,DEPOIS,IMPLANTE DE STENT CORONARIO COM OU SEM ANGIOPL...,1
6,356645,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1



Análise 3 concluída.


In [40]:
# =============================================================================
# Célula 6 (VERSÃO 3 - Corrigindo Múltiplos Registos de Infarto): Análise 4
# =============================================================================
import numpy as np
import pandas as pd

print("⏳ Iniciando Análise 4 (VERSÃO 3): Construindo a Linha do Tempo Real (em dias)...")

# 1. Garantir que as colunas de data e tipo de atendimento existem e estão corretas
try:
    df_jornada_infarto['DATA_ATENDIMENTO_FATO_PRO'] = pd.to_datetime(df_jornada_infarto['DATA_ATENDIMENTO_FATO_PRO'])

    for col in ['SERVICO', 'AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'DESCRICAO_CID']:
        if col in df_jornada_infarto.columns:
            df_jornada_infarto[col] = df_jornada_infarto[col].astype(str)

    print("  - Coluna de datas convertida para formato datetime.")
    print("  - Colunas de texto (incluindo SERVICO) garantidas como string.")

except Exception as e:
    print(f"Erro na preparação inicial: {e}.")

# 2. Encontrar o "Dia Zero": a primeira data de registro de 'INFARTO AGUDO' por paciente
mask_infarto_temp = df_jornada_infarto['DESCRICAO_CID'].str.contains('INFARTO AGUDO', case=False, na=False)
df_infarto_eventos = df_jornada_infarto[mask_infarto_temp]

event_0_dates = df_infarto_eventos.groupby('ID_PESSOA')['DATA_ATENDIMENTO_FATO_PRO'].min().reset_index()
event_0_dates.columns = ['ID_PESSOA', 'Data_Evento_0']

print("\n🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Infarto):")
display(event_0_dates)

# 3. Juntar a 'Data_Evento_0' a todos os registros dos pacientes
df_jornada_timeline = pd.merge(df_jornada_infarto, event_0_dates, on='ID_PESSOA', how='left')

# 4. Calcular os dias relativos ao "Dia Zero"
df_jornada_timeline['Dias_Desde_Evento_0'] = (df_jornada_timeline['DATA_ATENDIMENTO_FATO_PRO'] - df_jornada_timeline['Data_Evento_0']).dt.days

# 5. Definir as máscaras para TODOS os eventos-chave que queremos na timeline
try:
    pattern
except NameError:
    print("  - Recriando 'pattern' (keywords de procedimento)...")
    keywords_procedimentos = ['ANGIOTOMOGRAFIA CORONARIANA', 'CATETERISMO CARDIACO', 'IMPLANTE DE STENT CORONARIO']
    pattern = '|'.join(keywords_procedimentos)

print("  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline)...")

mask_proc = df_jornada_timeline['SERVICO'].str.contains(pattern, case=False, na=False)
mask_emerg = df_jornada_timeline['TIPO_ATEND'] == 'EMERGENCIA'
mask_cardio_elet = df_jornada_timeline['TIPO_ATEND'] == 'CARDIO_ELETIVA'
mask_infarto = df_jornada_timeline['DESCRICAO_CID'].str.contains('INFARTO AGUDO', case=False, na=False)

# 6. Filtrar o DataFrame para conter APENAS os eventos-chave
#    (Nota: Continuamos a incluir TODAS as linhas de infarto aqui...)
mask_key_events = mask_infarto | mask_proc | mask_emerg | mask_cardio_elet
df_timeline_filtrada = df_jornada_timeline[mask_key_events].copy()

# 7. Criar Etiquetas claras para os eventos (em ordem de prioridade)
#    (... e filtramos a etiqueta no passo 7!)
conditions = [
    df_timeline_filtrada['SERVICO'].str.contains('IMPLANTE DE STENT', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('CATETERISMO CARDIACO', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('ANGIOTOMOGRAFIA CORONARIANA', case=False, na=False),

    # --- A CORREÇÃO ESTÁ AQUI ---
    # Agora, a etiqueta "EVENTO: Registro Infarto" SÓ é aplicada se
    # a data do evento for a MESMA data do "Data_Evento_0" (o Dia Zero).
    (df_timeline_filtrada['DESCRICAO_CID'].str.contains('INFARTO AGUDO', case=False, na=False)) &
    (df_timeline_filtrada['DATA_ATENDIMENTO_FATO_PRO'] == df_timeline_filtrada['Data_Evento_0']),
    # -----------------------------

    df_timeline_filtrada['TIPO_ATEND'] == 'EMERGENCIA',
    df_timeline_filtrada['TIPO_ATEND'] == 'CARDIO_ELETIVA'
]

choices = [
    'PROC: Implante de Stent',
    'PROC: Cateterismo Cardiaco',
    'PROC: Angiotomografia COR',
    'EVENTO: Registro Infarto', # Esta escolha só é ativada pela nova condição dupla
    'VISITA: Emergência',
    'VISITA: Cardio Eletiva'
]

# O 'default' vai apanhar os outros dias de 'INFARTO AGUDO' que não são o Dia 0
df_timeline_filtrada['Evento_Label'] = np.select(conditions, choices, default='Outro (Reg. Infarto Sec.)')

# 8. Preparar e exibir a Tabela-Resumo Final
colunas_finais = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Dias_Desde_Evento_0', 'Evento_Label']
df_timeline_final_view = df_timeline_filtrada[colunas_finais]

# --- ADICIONAR UM FILTRO EXTRA ---
# Agora que aplicámos as etiquetas, podemos remover com segurança
# as linhas 'Outro' que não queremos ver, como os registos secundários de infarto.
df_timeline_final_view = df_timeline_final_view[df_timeline_final_view['Evento_Label'].str.contains('Outro') == False].copy()

# Ordenar e remover duplicatas
df_timeline_final_view = df_timeline_final_view.sort_values(by=['ID_PESSOA', 'Dias_Desde_Evento_0'])
df_timeline_final_view = df_timeline_final_view.drop_duplicates(subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Evento_Label'])

print("\n--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes) [VERSÃO 3 CORRIGIDA] ---")
print("(Apenas o PRIMEIRO registro de Infarto (Dia 0) é mostrado)")
display(df_timeline_final_view)

print("\n✅ Análise 4 (Versão 3) concluída. A cronologia está agora limpa.")

⏳ Iniciando Análise 4 (VERSÃO 3): Construindo a Linha do Tempo Real (em dias)...
  - Coluna de datas convertida para formato datetime.
  - Colunas de texto (incluindo SERVICO) garantidas como string.

🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Infarto):


,ID_PESSOA,Data_Evento_0
0,333908,2022-09-27
1,356645,2022-04-11


  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline)...

--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes) [VERSÃO 3 CORRIGIDA] ---
(Apenas o PRIMEIRO registro de Infarto (Dia 0) é mostrado)


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,Dias_Desde_Evento_0,Evento_Label
9,333908,2022-09-27,0,EVENTO: Registro Infarto
125,333908,2022-09-27,0,PROC: Cateterismo Cardiaco
218,333908,2022-09-27,0,VISITA: Emergência
58,333908,2022-10-07,10,PROC: Implante de Stent
219,333908,2022-11-28,62,VISITA: Emergência
220,333908,2022-12-31,95,VISITA: Emergência
221,333908,2023-03-14,168,VISITA: Emergência
222,333908,2023-04-30,215,VISITA: Emergência
223,333908,2024-05-18,599,VISITA: Emergência
224,333908,2024-10-08,742,VISITA: Emergência



✅ Análise 4 (Versão 3) concluída. A cronologia está agora limpa.


In [41]:
# =============================================================================
# Célula 7: Visualização da Cronologia (Timeline Plot)
# =============================================================================
import plotly.express as px

print("📊 Iniciando Célula 7: Geração do Gráfico de Cronologia...")

# --- Preparação para o Gráfico ---
# 1. É uma boa prática copiar o DataFrame final para não alterar o original
df_plot = df_timeline_final_view.copy()

# 2. Garantir que o ID do Paciente é tratado como texto (categoria) e não um número
# Isso faz com que cada paciente tenha a sua própria "linha" no eixo Y
df_plot['ID_PESSOA'] = df_plot['ID_PESSOA'].astype(str)

print("  - Dados preparados para a visualização.")

# --- Criação do Gráfico Interativo ---
fig = px.scatter(
    data_frame=df_plot,
    x='Dias_Desde_Evento_0',
    y='ID_PESSOA',
    color='Evento_Label',              # Define as cores com base na etiqueta do evento
    hover_data=[                     # O que mostrar ao passar o rato
        'DATA_ATENDIMENTO_FATO_PRO',
        'Evento_Label',
        'Dias_Desde_Evento_0'
    ],
    labels={                         # Renomear os eixos para ficarem claros
        'Dias_Desde_Evento_0': 'Dias desde o Evento Zero (1º Infarto)',
        'ID_PESSOA': 'Paciente',
        'Evento_Label': 'Tipo de Evento'
    },
    title='Cronologia da Jornada do Paciente (Dias desde o 1º Infarto)'
)

# --- Melhorias no Gráfico ---
# 1. Adicionar uma linha vertical vermelha no "Dia 0" para marcar o evento principal
fig.add_vline(
    x=0,
    line_width=2,
    line_dash="dash",
    line_color="red",
    annotation_text="Dia 0 (Infarto)", # Adiciona um texto à linha
    annotation_position="bottom right"
)

# 2. Garantir que o eixo Y é tratado como 'categoria' (embora o astype(str) já ajude)
fig.update_yaxes(type='category')

# 3. Ordenar o eixo Y (pacientes)
fig.update_layout(yaxis={'categoryorder':'total ascending'})

print("  - Gráfico gerado com sucesso.")

# --- Exibir o Gráfico ---
fig.show()

print("\n✅ Visualização concluída.")

📊 Iniciando Célula 7: Geração do Gráfico de Cronologia...
  - Dados preparados para a visualização.
  - Gráfico gerado com sucesso.



✅ Visualização concluída.


In [43]:
# =============================================================================
# Célula 8: Análise 5 - Cálculo de Métricas Chave (KPIs) da Jornada
# =============================================================================

print("📈 Iniciando Análise 5: Calculando KPIs da Jornada...")

# Vamos usar a nossa tabela-resumo final como base
df_base = df_timeline_final_view.copy()

# --- 1. KPI: Tempo (dias) até ao primeiro 'Implante de Stent' ---

# Filtra apenas os eventos de 'Implante de Stent'
df_stent = df_base[df_base['Evento_Label'] == 'PROC: Implante de Stent']

# Agrupamos por paciente e pegamos o primeiro dia (o menor) em que o evento ocorreu
# Se um paciente teve 2 stents (Dia 3 e Dia 100), isto irá pegar o "Dia 3"
kpi_stent = df_stent.groupby('ID_PESSOA')['Dias_Desde_Evento_0'].min().reset_index()

# Renomeamos a coluna para clareza
kpi_stent.columns = ['ID_PESSOA', 'Tempo_Ate_Primeiro_Stent (dias)']
print("  - KPI 1 (Tempo até Stent) calculado.")


# --- 2. KPI: Tempo (dias) até ao primeiro 'Cateterismo Cardiaco' (no Dia 0 ou depois) ---

# Filtra eventos de 'Cateterismo Cardiaco' QUE OCORRERAM NO DIA 0 OU DEPOIS
df_cateter = df_base[
    (df_base['Evento_Label'] == 'PROC: Cateterismo Cardiaco') &
    (df_base['Dias_Desde_Evento_0'] >= 0)
]

# Agrupamos por paciente e pegamos o primeiro dia (o menor)
kpi_cateter = df_cateter.groupby('ID_PESSOA')['Dias_Desde_Evento_0'].min().reset_index()

# Renomeamos a coluna
kpi_cateter.columns = ['ID_PESSOA', 'Tempo_Ate_Primeiro_Cateterismo (dias)']
print("  - KPI 2 (Tempo até Cateterismo) calculado.")


# --- 3. KPI: Número total de 'VISITA: Emergência' APÓS o Dia 0 ---

# Filtra visitas à emergência APENAS DEPOIS do Dia 0
df_emerg = df_base[
    (df_base['Evento_Label'] == 'VISITA: Emergência') &
    (df_base['Dias_Desde_Evento_0'] > 0) # > 0 significa "depois" do dia 0
]

# Agrupamos por paciente e contamos quantas linhas (.size()) existem para cada um
kpi_emerg = df_emerg.groupby('ID_PESSOA').size().reset_index(name='Qtd_Emergencias_Pos_Infarto')
print("  - KPI 3 (Emergências Pós-Infarto) calculado.")


# --- 4. Consolidar a tabela de KPIs ---

# Começamos com uma lista única de todos os pacientes da nossa análise
df_pacientes = df_base[['ID_PESSOA']].drop_duplicates()

# Juntamos os KPIs, um por um, usando 'how="left"'
# 'how="left"' garante que mantemos o paciente na lista mesmo que ele não tenha um KPI
# (ex: se um paciente não teve Stent, ele aparecerá com 'NaN' (Nulo))

df_kpi = df_pacientes.merge(kpi_stent, on='ID_PESSOA', how='left')
df_kpi = df_kpi.merge(kpi_cateter, on='ID_PESSOA', how='left')
df_kpi = df_kpi.merge(kpi_emerg, on='ID_PESSOA', how='left')

print("  - KPIs consolidados numa tabela final.")

# --- 5. Limpeza Final ---

# Se um paciente não teve emergências pós-infarto, o merge cria um 'NaN' (Nulo).
# Vamos substituir 'NaN' por 0, pois é mais correto dizer "0 visitas" do que "Nulo".
df_kpi['Qtd_Emergencias_Pos_Infarto'] = df_kpi['Qtd_Emergencias_Pos_Infarto'].fillna(0).astype(int)

# Nota: Para os KPIs de tempo, 'NaN' (Nulo) é bom, pois significa "Evento não ocorreu"

# --- 6. Exibir Resultados ---
print("\n--- 📊 Métricas Chave de Performance (KPIs) da Jornada ---")
display(df_kpi)

print("\n✅ Análise 5 (KPIs) concluída.")

📈 Iniciando Análise 5: Calculando KPIs da Jornada...
  - KPI 1 (Tempo até Stent) calculado.
  - KPI 2 (Tempo até Cateterismo) calculado.
  - KPI 3 (Emergências Pós-Infarto) calculado.
  - KPIs consolidados numa tabela final.

--- 📊 Métricas Chave de Performance (KPIs) da Jornada ---


,ID_PESSOA,Tempo_Ate_Primeiro_Stent (dias),Tempo_Ate_Primeiro_Cateterismo (dias),Qtd_Emergencias_Pos_Infarto
0,333908,10,0,8
1,356645,3,0,3



✅ Análise 5 (KPIs) concluída.


In [44]:
# =============================================================================
# Célula 9: Análise 6 - Cálculo de Métricas Preventivas (KPIs Pré-Infarto)
# =============================================================================

print("🚨 Iniciando Análise 6: Calculando KPIs Preventivos (Sinais de Alerta Pré-Infarto)...")

# 1. Criar um DataFrame focado APENAS em eventos ANTES do Dia 0
df_pre_infarto = df_timeline_final_view[
    df_timeline_final_view['Dias_Desde_Evento_0'] < 0
].copy()

if df_pre_infarto.empty:
    print("  - Nenhum evento registrado antes do 'Dia Zero' para os pacientes da amostra.")

else:
    print(f"  - {df_pre_infarto.shape[0]} eventos pré-infarto encontrados para análise.")


# --- 2. Calcular os KPIs Preventivos ---

# KPI 1: Qtd. Visitas Cardio Eletiva (Pré-Infarto)
df_cardio_pre = df_pre_infarto[
    df_pre_infarto['Evento_Label'] == 'VISITA: Cardio Eletiva'
]
kpi_cardio_pre = df_cardio_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Cardio_Eletiva_Pre_Infarto')

# KPI 2: Qtd. Visitas Emergência (Pré-Infarto)
df_emerg_pre = df_pre_infarto[
    df_pre_infarto['Evento_Label'] == 'VISITA: Emergência'
]
kpi_emerg_pre = df_emerg_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Emergencia_Pre_Infarto')

# KPI 3: Qtd. Procedimentos Chave (Pré-Infarto) - (Angio, Cateter, Stent)
df_proc_pre = df_pre_infarto[
    df_pre_infarto['Evento_Label'].str.contains('PROC:')
]
kpi_proc_pre = df_proc_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Procedimentos_Pre_Infarto')

print("  - KPIs preventivos calculados.")

# --- 3. Consolidar a tabela de KPIs Preventivos ---

# Começamos com a lista completa de pacientes da nossa análise
# (Podemos reutilizar a 'df_pacientes' da Célula 8, se esta célula for executada na mesma sessão)
try:
    df_pacientes
except NameError:
    print("  - Recriando lista de pacientes...")
    df_pacientes = df_timeline_final_view[['ID_PESSOA']].drop_duplicates()

# Juntar os KPIs preventivos
df_kpi_preventivo = df_pacientes.merge(kpi_cardio_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo = df_kpi_preventivo.merge(kpi_emerg_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo = df_kpi_preventivo.merge(kpi_proc_pre, on='ID_PESSOA', how='left')

# Substituir 'NaN' (Nulo) por 0. 'NaN' aqui significa "nenhum evento", ou seja, 0.
cols_kpi = ['Qtd_Cardio_Eletiva_Pre_Infarto', 'Qtd_Emergencia_Pre_Infarto', 'Qtd_Procedimentos_Pre_Infarto']
df_kpi_preventivo[cols_kpi] = df_kpi_preventivo[cols_kpi].fillna(0).astype(int)

print("  - KPIs preventivos consolidados.")

# --- 4. Exibir Resultados ---
print("\n--- 🚨 Métricas Preventivas (Comportamento Pré-Infarto) ---")
display(df_kpi_preventivo)

print("\n✅ Análise 6 (KPIs Preventivos) concluída.")

🚨 Iniciando Análise 6: Calculando KPIs Preventivos (Sinais de Alerta Pré-Infarto)...
  - 2 eventos pré-infarto encontrados para análise.
  - KPIs preventivos calculados.
  - KPIs preventivos consolidados.

--- 🚨 Métricas Preventivas (Comportamento Pré-Infarto) ---


,ID_PESSOA,Qtd_Cardio_Eletiva_Pre_Infarto,Qtd_Emergencia_Pre_Infarto,Qtd_Procedimentos_Pre_Infarto
0,333908,0,0,0
1,356645,2,0,0



✅ Análise 6 (KPIs Preventivos) concluída.


In [45]:
# =============================================================================
# Célula 4 (VERSÃO ANGINA): Jornada completa dos pacientes que tiveram ANGINA
# (Versão com resumos principais)
# =============================================================================
import pandas as pd
import numpy as np

print("🔍 Iniciando análise: jornada completa dos pacientes com ANGINA...")

# 1) identificar IDs que em algum momento tiveram ANGINA (qualquer linha)
# --- CORREÇÃO PRINCIPAL AQUI ---
keywords_angina = [
    'ANGINA INSTAVEL',
    'ANGINA PECTORIS, NAO ESPECIFICADA',
    'ANGINA PECTORIS', # Adicionado por segurança
    'OUTRAS FORMAS DE ANGINA PECTORIS'
]
pattern_angina = '|'.join(keywords_angina)
print(f"  - Padrão de busca: {pattern_angina}")

mask_angina_any = df['DESCRICAO_CID'].str.contains(pattern_angina, case=False, na=False)
ids_angina = df.loc[mask_angina_any, 'ID_PESSOA'].unique()
print(f"🫀 IDs únicos com registro de ANGINA: {len(ids_angina)}")

# 2) obter todas as linhas desses pacientes (toda a jornada)
#    (Renomeado para df_jornada_angina para clareza)
df_jornada_angina = df[df['ID_PESSOA'].isin(ids_angina)].copy()
print(f"📂 Registros desses pacientes (toda jornada): {df_jornada_angina.shape[0]:,}")

# 3) garantir colunas de interesse como texto (evita problemas de NaN/type)
for c in ['AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'MOMENTO', 'DESCRICAO_CID']:
    df_jornada_angina[c] = df_jornada_angina[c].astype(str)

# 4) separar antes/depois pelo campo MOMENTO
antes = df_jornada_angina[df_jornada_angina['MOMENTO'].str.upper().str.strip() == 'ANTES']
depois = df_jornada_angina[df_jornada_angina['MOMENTO'].str.upper().str.strip() == 'DEPOIS']
print(f"  - Registros 'Antes': {antes.shape[0]}. Registros 'Depois': {depois.shape[0]}.")

# 5) métricas desejadas: (LÓGICA SIMPLIFICADA)
#    (As máscaras agora são aplicadas diretamente em 'antes' e 'depois')

# --- Métricas do período ANTES ---
mask_emerg_antes = antes['AGRUP_ASSISTENCIAL_G'].str.contains('EMERGENCIA', case=False, na=False)
mask_eletiva_antes = antes['AGRUP_ASSISTENCIAL_G'].str.contains('ELETIVA', case=False, na=False)
mask_cardio_antes = antes['ESPECIALIDADE'].str.contains('CARDIO', case=False, na=False)

emerg_antes_total = antes[mask_emerg_antes].shape[0]
emerg_antes_pacientes = antes[mask_emerg_antes]['ID_PESSOA'].nunique()
cardio_elet_antes_total = antes[mask_eletiva_antes & mask_cardio_antes].shape[0]
cardio_elet_antes_pacientes = antes[mask_eletiva_antes & mask_cardio_antes]['ID_PESSOA'].nunique()

# --- Métricas do período DEPOIS ---
mask_emerg_depois = depois['AGRUP_ASSISTENCIAL_G'].str.contains('EMERGENCIA', case=False, na=False)
mask_eletiva_depois = depois['AGRUP_ASSISTENCIAL_G'].str.contains('ELETIVA', case=False, na=False)
mask_cardio_depois = depois['ESPECIALIDADE'].str.contains('CARDIO', case=False, na=False)

emerg_depois_total = depois[mask_emerg_depois].shape[0]
emerg_depois_pacientes = depois[mask_emerg_depois]['ID_PESSOA'].nunique()
cardio_elet_depois_total = depois[mask_eletiva_depois & mask_cardio_depois].shape[0]
cardio_elet_depois_pacientes = depois[mask_eletiva_depois & mask_cardio_depois]['ID_PESSOA'].nunique()
print("  - Métricas 'Antes' e 'Depois' calculadas.")

# 6) montar resumo
resumo = pd.DataFrame({
    'Período': ['Antes', 'Depois'],
    'Pacientes únicos (nesse período)': [antes['ID_PESSOA'].nunique(), depois['ID_PESSOA'].nunique()],
    'Atendimentos emergência (total)': [emerg_antes_total, emerg_depois_total],
    'Pacientes com emergência': [emerg_antes_pacientes, emerg_depois_pacientes],
    'Consultas cardio eletiva (total)': [cardio_elet_antes_total, cardio_elet_depois_total],
    'Pacientes com cardio eletiva': [cardio_elet_antes_pacientes, cardio_elet_depois_pacientes]
})
print("\n📊 Resumo da Jornada (Angina):")
display(resumo)

# 7) Top especialidades (antes / depois)
esp_antes = antes['ESPECIALIDADE'].value_counts().reset_index()
esp_antes.columns = ['Especialidade', 'Qtd_Antes']

esp_depois = depois['ESPECIALIDADE'].value_counts().reset_index()
esp_depois.columns = ['Especialidade', 'Qtd_Depois']

esp_comparativo = pd.merge(esp_antes, esp_depois, on='Especialidade', how='outer').fillna(0)
esp_comparativo = esp_comparativo.sort_values(by=['Qtd_Antes','Qtd_Depois'], ascending=False)
print("\n📋 Especialidades (antes/depois) — tabela completa (ordenada):")
display(esp_comparativo.head(50))

# 8) Linha do tempo por paciente: contagem de eventos por MOMENTO e tipo
def tipo_atendimento(row):
    # (Garante que colunas são string antes de usar .upper())
    a = str(row['AGRUP_ASSISTENCIAL_G']).upper()
    e = str(row['ESPECIALIDADE']).upper()
    if 'EMERGENCIA' in a: return 'EMERGENCIA'
    if 'ELETIVA' in a and 'CARDIO' in e: return 'CARDIO_ELETIVA'
    if 'ELETIVA' in a: return 'ELETIVA_OUTRA'
    return 'OUTRO'

df_jornada_angina['TIPO_ATEND'] = df_jornada_angina.apply(tipo_atendimento, axis=1)

timeline = df_jornada_angina.groupby(['ID_PESSOA','MOMENTO','TIPO_ATEND']).size().unstack(fill_value=0).reset_index()
print("\n🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):")
display(timeline.head(50))

print("\n✅ Análise da jornada de ANGINA executada. Apenas os resumos principais foram exibidos.")

🔍 Iniciando análise: jornada completa dos pacientes com ANGINA...
  - Padrão de busca: ANGINA INSTAVEL|ANGINA PECTORIS, NAO ESPECIFICADA|ANGINA PECTORIS|OUTRAS FORMAS DE ANGINA PECTORIS
🫀 IDs únicos com registro de ANGINA: 64
📂 Registros desses pacientes (toda jornada): 23,984
  - Registros 'Antes': 13330. Registros 'Depois': 10055.
  - Métricas 'Antes' e 'Depois' calculadas.

📊 Resumo da Jornada (Angina):


,Período,Pacientes únicos (nesse período),Atendimentos emergência (total),Pacientes com emergência,Consultas cardio eletiva (total),Pacientes com cardio eletiva
0,Antes,64,110,42,170,52
1,Depois,64,81,33,151,50



📋 Especialidades (antes/depois) — tabela completa (ordenada):


,Especialidade,Qtd_Antes,Qtd_Depois
39,NAO INFORMADO,8627.0,7474.0
6,CARDIOLOGIA,814.0,450.0
24,FISIOTERAPIA,666.0,460.0
46,OFTALMOLOGIA,433.0,214.0
57,RADIOLOGIA,404.0,171.0
17,CLINICA MEDICA,350.0,328.0
7,CIRURGIA CARDIOVASCULAR,298.0,44.0
50,PATOLOGIA,271.0,67.0
18,DERMATOLOGIA,178.0,41.0
49,OTORRINOLARINGOLOGIA,149.0,88.0



🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):


TIPO_ATEND,ID_PESSOA,MOMENTO,CARDIO_ELETIVA,ELETIVA_OUTRA,EMERGENCIA,OUTRO
0,39514,ANTES,0,2,1,39
1,39514,DEPOIS,0,2,1,115
2,39514,MESMA DATA,0,0,1,19
3,73051,ANTES,6,14,0,173
4,73051,DEPOIS,5,5,0,106
5,73051,MESMA DATA,0,0,0,12
6,96146,ANTES,0,4,2,59
7,96146,DEPOIS,1,1,0,68
8,96146,MESMA DATA,0,0,0,2
9,104531,ANTES,4,7,0,127



✅ Análise da jornada de ANGINA executada. Apenas os resumos principais foram exibidos.


In [46]:
# =============================================================================
# Célula 5 (VERSÃO ANGINA): Análise 3 - Busca por procedimentos-chave
# =============================================================================
print("Iniciando Célula 5 (para Angina): Busca por procedimentos-chave (Angio/Cateter/Stent)...")

# 1. Palavras-chave dos procedimentos (são as mesmas para Angina)
keywords_procedimentos = [
    'ANGIOTOMOGRAFIA CORONARIANA',
    'CATETERISMO CARDIACO',
    'IMPLANTE DE STENT CORONARIO'
]

# 2. Padrão regex com OU
#    (Importante: Esta variável 'pattern' será reutilizada na Célula 6)
pattern = '|'.join(keywords_procedimentos)
print(f"  - Padrão de busca por procedimentos: {pattern}")

# 3. Filtrar registros com os procedimentos
#    --- ESTA É A MUDANÇA ---
#    Agora procuramos no DataFrame df_jornada_angina
try:
    df_jornada_angina['SERVICO'] = df_jornada_angina['SERVICO'].astype(str)
    mask = df_jornada_angina['SERVICO'].str.contains(pattern, case=False, na=False)
    df_proc = df_jornada_angina[mask].copy()
    print("  - Filtro de procedimentos aplicado ao df_jornada_angina.")
except Exception as e:
    print(f"ERRO: O DataFrame 'df_jornada_angina' não foi encontrado ou está sem a coluna 'SERVICO'. {e}")
    # Criar um DataFrame vazio para o código não quebrar
    df_proc = pd.DataFrame(columns=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'SERVICO', 'MOMENTO'])

# 4. REMOVER DUPLICATAS: mesmo paciente + mesma data + mesmo serviço = 1 evento
df_proc_unicos = df_proc.drop_duplicates(
    subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'SERVICO']
)

# 5. Selecionar apenas colunas relevantes
colunas_display = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'MOMENTO', 'SERVICO']

# 6. Resultados
if df_proc_unicos.empty:
    print("\nRESULTADO: Nenhum procedimento-chave encontrado para os pacientes com Angina.")
else:
    print(f"\nSUCESSO! Encontrados {df_proc_unicos.shape[0]} procedimentos-chave únicos.\n")

    print("-- Detalhe dos Procedimentos (1 por data/serviço) --")
    display(df_proc_unicos[colunas_display]
            .sort_values(by=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO']))

    print("\n-- Resumo por Paciente e Momento --")
    resumo = (df_proc_unicos
                .groupby(['ID_PESSOA', 'MOMENTO', 'SERVICO'])
                .size()
                .reset_index(name='Qtd_Eventos'))
    display(resumo)

print("\nAnálise 3 (Célula 5) concluída para Angina.")

Iniciando Célula 5 (para Angina): Busca por procedimentos-chave (Angio/Cateter/Stent)...
  - Padrão de busca por procedimentos: ANGIOTOMOGRAFIA CORONARIANA|CATETERISMO CARDIACO|IMPLANTE DE STENT CORONARIO
  - Filtro de procedimentos aplicado ao df_jornada_angina.

SUCESSO! Encontrados 142 procedimentos-chave únicos.

-- Detalhe dos Procedimentos (1 por data/serviço) --


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,MOMENTO,SERVICO
10161,39514,2023-09-25,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
37943,73051,2024-04-10,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
37729,73051,2024-04-25,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
37860,73051,2024-05-21,DEPOIS,IMPLANTE DE STENT CORONARIO COM OU SEM ANGIOPL...
58011,96146,2025-04-15,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
...,...,...,...,...
590265,1155280,2025-01-02,ANTES,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
590582,1155280,2025-01-14,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
591824,1170914,2025-02-19,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
591716,1170914,2025-03-06,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...



-- Resumo por Paciente e Momento --


,ID_PESSOA,MOMENTO,SERVICO,Qtd_Eventos
0,39514,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
1,73051,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
2,73051,DEPOIS,IMPLANTE DE STENT CORONARIO COM OU SEM ANGIOPL...,1
3,73051,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
4,96146,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
...,...,...,...,...
130,1155280,ANTES,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,2
131,1155280,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
132,1170914,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
133,1170914,DEPOIS,IMPLANTE DE STENT CORONARIO COM OU SEM ANGIOPL...,1



Análise 3 (Célula 5) concluída para Angina.


In [47]:
# =============================================================================
# Célula 6 (VERSÃO ANGINA): Análise 4 - Cronologia Real (Timeline em Dias)
# =============================================================================
import numpy as np
import pandas as pd

print("⏳ Iniciando Análise 4 (VERSÃO ANGINA): Construindo a Linha do Tempo Real (em dias)...")

# 0. Definir o DataFrame de base
df_base_angina = df_jornada_angina.copy()

# 1. Garantir que as colunas de data e tipo de atendimento existem e estão corretas
try:
    df_base_angina['DATA_ATENDIMENTO_FATO_PRO'] = pd.to_datetime(df_base_angina['DATA_ATENDIMENTO_FATO_PRO'])

    # Garantir que colunas de texto são strings
    for col in ['SERVICO', 'AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'DESCRICAO_CID']:
        if col in df_base_angina.columns:
            df_base_angina[col] = df_base_angina[col].astype(str)

    print("  - Coluna de datas convertida para formato datetime.")
    print("  - Colunas de texto (incluindo SERVICO) garantidas como string.")

except Exception as e:
    print(f"Erro na preparação inicial: {e}.")

# (Opcional) Recriar TIPO_ATEND por segurança, caso a Célula 4 não tenha sido executada
if 'TIPO_ATEND' not in df_base_angina.columns:
    print("  - Recriando coluna 'TIPO_ATEND'...")
    def tipo_atendimento(row):
        a = str(row['AGRUP_ASSISTENCIAL_G']).upper()
        e = str(row['ESPECIALIDADE']).upper()
        if 'EMERGENCIA' in a: return 'EMERGENCIA'
        if 'ELETIVA' in a and 'CARDIO' in e: return 'CARDIO_ELETIVA'
        if 'ELETIVA' in a: return 'ELETIVA_OUTRA'
        return 'OUTRO'
    df_base_angina['TIPO_ATEND'] = df_base_angina.apply(tipo_atendimento, axis=1)


# 2. Encontrar o "Dia Zero": a primeira data de registro de 'ANGINA' por paciente
#    (Verificar se 'pattern_angina' existe da Célula 4)
try:
    pattern_angina
except NameError:
    print("  - Recriando 'pattern_angina' (keywords de CID)...")
    keywords_angina = [
        'ANGINA INSTAVEL',
        'ANGINA PECTORIS, NAO ESPECIFICADA',
        'ANGINA PECTORIS',
        'OUTRAS FORMAS DE ANGINA PECTORIS'
    ]
    pattern_angina = '|'.join(keywords_angina)

mask_angina_temp = df_base_angina['DESCRICAO_CID'].str.contains(pattern_angina, case=False, na=False)
df_angina_eventos = df_base_angina[mask_angina_temp]

event_0_dates = df_angina_eventos.groupby('ID_PESSOA')['DATA_ATENDIMENTO_FATO_PRO'].min().reset_index()
event_0_dates.columns = ['ID_PESSOA', 'Data_Evento_0']

print("\n🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Angina):")
display(event_0_dates)

# 3. Juntar a 'Data_Evento_0' a todos os registros dos pacientes
df_jornada_timeline_angina = pd.merge(df_base_angina, event_0_dates, on='ID_PESSOA', how='left')

# 4. Calcular os dias relativos ao "Dia Zero"
df_jornada_timeline_angina['Dias_Desde_Evento_0'] = (df_jornada_timeline_angina['DATA_ATENDIMENTO_FATO_PRO'] - df_jornada_timeline_angina['Data_Evento_0']).dt.days

# 5. Definir as máscaras para TODOS os eventos-chave
#    (Verificar se 'pattern' de procedimentos existe da Célula 5)
try:
    pattern
except NameError:
    print("  - Recriando 'pattern' (keywords de procedimento)...")
    keywords_procedimentos = ['ANGIOTOMOGRAFIA CORONARIANA', 'CATETERISMO CARDIACO', 'IMPLANTE DE STENT CORONARIO']
    pattern = '|'.join(keywords_procedimentos)

print("  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline_angina)...")

# Todas as máscaras são definidas no df_jornada_timeline_angina
mask_proc = df_jornada_timeline_angina['SERVICO'].str.contains(pattern, case=False, na=False)
mask_emerg = df_jornada_timeline_angina['TIPO_ATEND'] == 'EMERGENCIA'
mask_cardio_elet = df_jornada_timeline_angina['TIPO_ATEND'] == 'CARDIO_ELETIVA'
# Recriamos a máscara de angina no DataFrame correto
mask_angina = df_jornada_timeline_angina['DESCRICAO_CID'].str.contains(pattern_angina, case=False, na=False)


# 6. Filtrar o DataFrame para conter APENAS os eventos-chave
mask_key_events = mask_angina | mask_proc | mask_emerg | mask_cardio_elet
df_timeline_filtrada = df_jornada_timeline_angina[mask_key_events].copy()

# 7. Criar Etiquetas claras para os eventos (em ordem de prioridade)
conditions = [
    df_timeline_filtrada['SERVICO'].str.contains('IMPLANTE DE STENT', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('CATETERISMO CARDIACO', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('ANGIOTOMOGRAFIA CORONARIANA', case=False, na=False),

    # --- A CORREÇÃO ESTÁ AQUI ---
    # Etiqueta "EVENTO" só é aplicada no "Dia Zero"
    (df_timeline_filtrada['DESCRICAO_CID'].str.contains(pattern_angina, case=False, na=False)) &
    (df_timeline_filtrada['DATA_ATENDIMENTO_FATO_PRO'] == df_timeline_filtrada['Data_Evento_0']),
    # -----------------------------

    df_timeline_filtrada['TIPO_ATEND'] == 'EMERGENCIA',
    df_timeline_filtrada['TIPO_ATEND'] == 'CARDIO_ELETIVA'
]

choices = [
    'PROC: Implante de Stent',
    'PROC: Cateterismo Cardiaco',
    'PROC: Angiotomografia COR',
    'EVENTO: Registro Angina', # Etiqueta atualizada
    'VISITA: Emergência',
    'VISITA: Cardio Eletiva'
]

df_timeline_filtrada['Evento_Label'] = np.select(conditions, choices, default='Outro (Reg. Angina Sec.)')

# 8. Preparar e exibir a Tabela-Resumo Final
colunas_finais = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Dias_Desde_Evento_0', 'Evento_Label']
df_timeline_final_view_angina = df_timeline_filtrada[colunas_finais]

# Remover os registros secundários (os 'Outros')
df_timeline_final_view_angina = df_timeline_final_view_angina[
    df_timeline_final_view_angina['Evento_Label'].str.contains('Outro') == False
].copy()

# Ordenar e remover duplicatas
df_timeline_final_view_angina = df_timeline_final_view_angina.sort_values(by=['ID_PESSOA', 'Dias_Desde_Evento_0'])
df_timeline_final_view_angina = df_timeline_final_view_angina.drop_duplicates(
    subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Evento_Label']
)

print("\n--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes com ANGINA) ---")
print("(Apenas o PRIMEIRO registro de Angina é mostrado como 'EVENTO')")
display(df_timeline_final_view_angina)

print("\n✅ Análise 4 (Célula 6) para ANGINA concluída. A cronologia está agora limpa.")

⏳ Iniciando Análise 4 (VERSÃO ANGINA): Construindo a Linha do Tempo Real (em dias)...
  - Coluna de datas convertida para formato datetime.
  - Colunas de texto (incluindo SERVICO) garantidas como string.

🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Angina):


,ID_PESSOA,Data_Evento_0
0,39514,2023-09-25
1,73051,2024-04-25
2,96146,2025-04-24
3,104531,2022-03-09
4,116885,2024-05-27
...,...,...
59,1022316,2024-05-29
60,1091370,2022-10-17
61,1150807,2024-07-10
62,1155280,2024-07-23


  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline_angina)...

--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes com ANGINA) ---
(Apenas o PRIMEIRO registro de Angina é mostrado como 'EVENTO')


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,Dias_Desde_Evento_0,Evento_Label
56,39514,2022-04-29,-514,VISITA: Emergência
57,39514,2023-09-25,0,VISITA: Emergência
67,39514,2023-09-25,0,EVENTO: Registro Angina
149,39514,2023-09-25,0,PROC: Angiotomografia COR
58,39514,2025-05-26,609,VISITA: Emergência
...,...,...,...,...
23932,1170914,2025-03-06,0,PROC: Implante de Stent
23875,1170914,2025-03-12,6,VISITA: Cardio Eletiva
23876,1170914,2025-03-27,21,VISITA: Cardio Eletiva
23878,1170914,2025-05-12,67,VISITA: Cardio Eletiva



✅ Análise 4 (Célula 6) para ANGINA concluída. A cronologia está agora limpa.


In [48]:
# =============================================================================
# Célula 7 (VERSÃO ANGINA): Visualização da Cronologia (Timeline Plot)
# =============================================================================
import plotly.express as px

print("📊 Iniciando Célula 7 (VERSÃO ANGINA): Geração do Gráfico de Cronologia...")

# --- Preparação para o Gráfico ---
# 1. Usar o DataFrame final da ANGINA
df_plot = df_timeline_final_view_angina.copy()

# 2. Garantir que o ID do Paciente é tratado como texto (categoria)
df_plot['ID_PESSOA'] = df_plot['ID_PESSOA'].astype(str)

# --- Calcular o Range Completo do Eixo X ---
# Vamos encontrar o dia mínimo (ex: -514) e o máximo
min_dias = df_plot['Dias_Desde_Evento_0'].min()
max_dias = df_plot['Dias_Desde_Evento_0'].max()

# Adicionar uma "margem" visual (ex: 30 dias)
min_dias_com_margem = min_dias - 30
max_dias_com_margem = max_dias + 30

print(f"  - Dados preparados. Range do eixo X definido de {min_dias_com_margem} até {max_dias_com_margem} dias.")

# --- Criação do Gráfico Interativo ---
fig = px.scatter(
    data_frame=df_plot,
    x='Dias_Desde_Evento_0',
    y='ID_PESSOA',
    color='Evento_Label',                  # Define as cores com base na etiqueta do evento
    hover_data=[                         # O que mostrar ao passar o rato
        'DATA_ATENDIMENTO_FATO_PRO',
        'Evento_Label',
        'Dias_Desde_Evento_0'
    ],
    labels={                             # Renomear os eixos para ficarem claros
        'Dias_Desde_Evento_0': 'Dias desde o Evento Zero (1ª Angina)',
        'ID_PESSOA': 'Paciente',
        'Evento_Label': 'Tipo de Evento'
    },
    # Título atualizado
    title='Cronologia da Jornada do Paciente (Dias desde a 1ª Angina)'
)

# --- Melhorias no Gráfico ---
# 1. Adicionar uma linha vertical vermelha no "Dia 0"
fig.add_vline(
    x=0,
    line_width=2,
    line_dash="dash",
    line_color="red",
    annotation_text="Dia 0 (Angina)", # Texto atualizado
    annotation_position="bottom right"
)

# 2. Melhorar a visualização para muitos pacientes
fig.update_yaxes(
    type='category',
    # Isto impede que o Plotly mostre todos os 64 IDs no eixo Y (ficaria ilegível)
    # Podes passar o rato por cima para ver o ID
    showticklabels=False,
    title='Pacientes (64 no total)' # Título do eixo Y atualizado
)

# 3. Ordenar o eixo Y (pacientes) E APLICAR O NOVO RANGE DO EIXO X
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_range=[min_dias_com_margem, max_dias_com_margem],
    height=800 # Aumentar a altura do gráfico para acomodar os 64 pacientes
)

print("  - Gráfico gerado com sucesso.")

# --- Exibir o Gráfico ---
fig.show()

print("\n✅ Visualização (Angina) concluída.")

📊 Iniciando Célula 7 (VERSÃO ANGINA): Geração do Gráfico de Cronologia...
  - Dados preparados. Range do eixo X definido de -1175 até 1459 dias.
  - Gráfico gerado com sucesso.



✅ Visualização (Angina) concluída.


In [49]:
# =============================================================================
# Célula 9 (VERSÃO ANGINA): Análise 6 - Cálculo de Métricas Preventivas
# =============================================================================

print("🚨 Iniciando Análise 6 (VERSÃO ANGINA): Calculando KPIs Preventivos (Sinais de Alerta Pré-Angina)...")

# 1. Criar um DataFrame focado APENAS em eventos ANTES do Dia 0
#    --- ESTA É A MUDANÇA ---
df_pre_angina = df_timeline_final_view_angina[
    df_timeline_final_view_angina['Dias_Desde_Evento_0'] < 0
].copy()

if df_pre_angina.empty:
    print("  - Nenhum evento registrado antes do 'Dia Zero' para os pacientes com Angina.")

else:
    print(f"  - {df_pre_angina.shape[0]} eventos pré-angina encontrados para análise.")


# --- 2. Calcular os KPIs Preventivos ---
# (A lógica de filtro é a mesma, estamos apenas a aplicá-la ao df_pre_angina)

# KPI 1: Qtd. Visitas Cardio Eletiva (Pré-Angina)
df_cardio_pre = df_pre_angina[
    df_pre_angina['Evento_Label'] == 'VISITA: Cardio Eletiva'
]
kpi_cardio_pre = df_cardio_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Cardio_Eletiva_Pre_Angina')

# KPI 2: Qtd. Visitas Emergência (Pré-Angina)
df_emerg_pre = df_pre_angina[
    df_pre_angina['Evento_Label'] == 'VISITA: Emergência'
]
kpi_emerg_pre = df_emerg_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Emergencia_Pre_Angina')

# KPI 3: Qtd. Procedimentos Chave (Pré-Angina) - (Angio, Cateter, Stent)
df_proc_pre = df_pre_angina[
    df_pre_angina['Evento_Label'].str.contains('PROC:')
]
kpi_proc_pre = df_proc_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Procedimentos_Pre_Angina')

print("  - KPIs preventivos calculados.")

# --- 3. Consolidar a tabela de KPIs Preventivos ---

# Começamos com a lista completa dos 64 pacientes de Angina
df_pacientes_angina = df_timeline_final_view_angina[['ID_PESSOA']].drop_duplicates()

# Juntar os KPIs preventivos
df_kpi_preventivo_angina = df_pacientes_angina.merge(kpi_cardio_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo_angina = df_kpi_preventivo_angina.merge(kpi_emerg_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo_angina = df_kpi_preventivo_angina.merge(kpi_proc_pre, on='ID_PESSOA', how='left')

# Substituir 'NaN' (Nulo) por 0. 'NaN' aqui significa "nenhum evento", ou seja, 0.
cols_kpi = ['Qtd_Cardio_Eletiva_Pre_Angina', 'Qtd_Emergencia_Pre_Angina', 'Qtd_Procedimentos_Pre_Angina']
df_kpi_preventivo_angina[cols_kpi] = df_kpi_preventivo_angina[cols_kpi].fillna(0).astype(int)

print("  - KPIs preventivos consolidados.")

# --- 4. Exibir Resultados ---
print("\n--- 🚨 Métricas Preventivas (Comportamento Pré-Angina) ---")
print("(Contagem de eventos ANTES do 'Dia Zero' de cada paciente)")
display(df_kpi_preventivo_angina)

# --- 5. (NOVO) Calcular Médias Gerais ---
print("\n--- 📊 Resumo de Métricas Preventivas (Médias) ---")
media_cardio = df_kpi_preventivo_angina['Qtd_Cardio_Eletiva_Pre_Angina'].mean()
media_emerg = df_kpi_preventivo_angina['Qtd_Emergencia_Pre_Angina'].mean()
media_proc = df_kpi_preventivo_angina['Qtd_Procedimentos_Pre_Angina'].mean()

print(f"  - Média de Consultas Cardio Eletivas (Pré-Angina): {media_cardio:.2f} por paciente")
print(f"  - Média de Visitas à Emergência (Pré-Angina): {media_emerg:.2f} por paciente")
print(f"  - Média de Procedimentos-Chave (Pré-Angina): {media_proc:.2f} por paciente")


print("\n✅ Análise 6 (KPIs Preventivos para Angina) concluída.")

🚨 Iniciando Análise 6 (VERSÃO ANGINA): Calculando KPIs Preventivos (Sinais de Alerta Pré-Angina)...
  - 228 eventos pré-angina encontrados para análise.
  - KPIs preventivos calculados.
  - KPIs preventivos consolidados.

--- 🚨 Métricas Preventivas (Comportamento Pré-Angina) ---
(Contagem de eventos ANTES do 'Dia Zero' de cada paciente)


,ID_PESSOA,Qtd_Cardio_Eletiva_Pre_Angina,Qtd_Emergencia_Pre_Angina,Qtd_Procedimentos_Pre_Angina
0,39514,0,1,0
1,73051,6,0,1
2,96146,1,2,1
3,104531,0,0,0
4,116885,4,0,1
...,...,...,...,...
59,1022316,1,1,1
60,1091370,0,0,0
61,1150807,1,0,1
62,1155280,3,6,1



--- 📊 Resumo de Métricas Preventivas (Médias) ---
  - Média de Consultas Cardio Eletivas (Pré-Angina): 1.80 por paciente
  - Média de Visitas à Emergência (Pré-Angina): 1.09 por paciente
  - Média de Procedimentos-Chave (Pré-Angina): 0.67 por paciente

✅ Análise 6 (KPIs Preventivos para Angina) concluída.


In [50]:
# =============================================================================
# Célula 4 (VERSÃO ARRITMIA): Jornada completa dos pacientes com ARRITMIA
# =============================================================================
import pandas as pd
import numpy as np

print("🔍 Iniciando análise: jornada completa dos pacientes com ARRITMIA...")

# 1) identificar IDs que em algum momento tiveram ARRITMIA (qualquer linha)
# --- NOVAS PALAVRAS-CHAVE ---
keywords_arritmia = [
    'ARRITMIA CARDIACA NAO ESPECIFICADA',
    'OUTRAS ARRITMIAS CARDÍACAS'
]
pattern_arritmia = '|'.join(keywords_arritmia)
print(f"  - Padrão de busca: {pattern_arritmia}")

mask_arritmia_any = df['DESCRICAO_CID'].str.contains(pattern_arritmia, case=False, na=False)
ids_arritmia = df.loc[mask_arritmia_any, 'ID_PESSOA'].unique()
print(f"🫀 IDs únicos com registro de ARRITMIA: {len(ids_arritmia)}")

# 2) obter todas as linhas desses pacientes (toda a jornada)
#    (Renomeado para df_jornada_arritmia)
df_jornada_arritmia = df[df['ID_PESSOA'].isin(ids_arritmia)].copy()
print(f"📂 Registros desses pacientes (toda jornada): {df_jornada_arritmia.shape[0]:,}")

# 3) garantir colunas de interesse como texto
for c in ['AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'MOMENTO', 'DESCRICAO_CID']:
    df_jornada_arritmia[c] = df_jornada_arritmia[c].astype(str)

# 4) separar antes/depois pelo campo MOMENTO
antes = df_jornada_arritmia[df_jornada_arritmia['MOMENTO'].str.upper().str.strip() == 'ANTES']
depois = df_jornada_arritmia[df_jornada_arritmia['MOMENTO'].str.upper().str.strip() == 'DEPOIS']
print(f"  - Registros 'Antes': {antes.shape[0]}. Registros 'Depois': {depois.shape[0]}.")

# 5) métricas desejadas:

# --- Métricas do período ANTES ---
mask_emerg_antes = antes['AGRUP_ASSISTENCIAL_G'].str.contains('EMERGENCIA', case=False, na=False)
mask_eletiva_antes = antes['AGRUP_ASSISTENCIAL_G'].str.contains('ELETIVA', case=False, na=False)
mask_cardio_antes = antes['ESPECIALIDADE'].str.contains('CARDIO', case=False, na=False)

emerg_antes_total = antes[mask_emerg_antes].shape[0]
emerg_antes_pacientes = antes[mask_emerg_antes]['ID_PESSOA'].nunique()
cardio_elet_antes_total = antes[mask_eletiva_antes & mask_cardio_antes].shape[0]
cardio_elet_antes_pacientes = antes[mask_eletiva_antes & mask_cardio_antes]['ID_PESSOA'].nunique()

# --- Métricas do período DEPOIS ---
mask_emerg_depois = depois['AGRUP_ASSISTENCIAL_G'].str.contains('EMERGENCIA', case=False, na=False)
mask_eletiva_depois = depois['AGRUP_ASSISTENCIAL_G'].str.contains('ELETIVA', case=False, na=False)
mask_cardio_depois = depois['ESPECIALIDADE'].str.contains('CARDIO', case=False, na=False)

emerg_depois_total = depois[mask_emerg_depois].shape[0]
emerg_depois_pacientes = depois[mask_emerg_depois]['ID_PESSOA'].nunique()
cardio_elet_depois_total = depois[mask_eletiva_depois & mask_cardio_depois].shape[0]
cardio_elet_depois_pacientes = depois[mask_eletiva_depois & mask_cardio_depois]['ID_PESSOA'].nunique()
print("  - Métricas 'Antes' e 'Depois' calculadas.")

# 6) montar resumo
resumo = pd.DataFrame({
    'Período': ['Antes', 'Depois'],
    'Pacientes únicos (nesse período)': [antes['ID_PESSOA'].nunique(), depois['ID_PESSOA'].nunique()],
    'Atendimentos emergência (total)': [emerg_antes_total, emerg_depois_total],
    'Pacientes com emergência': [emerg_antes_pacientes, emerg_depois_pacientes],
    'Consultas cardio eletiva (total)': [cardio_elet_antes_total, cardio_elet_depois_total],
    'Pacientes com cardio eletiva': [cardio_elet_antes_pacientes, cardio_elet_depois_pacientes]
})
print("\n📊 Resumo da Jornada (Arritmia):")
display(resumo)

# 7) Top especialidades (antes / depois)
esp_antes = antes['ESPECIALIDADE'].value_counts().reset_index()
esp_antes.columns = ['Especialidade', 'Qtd_Antes']

esp_depois = depois['ESPECIALIDADE'].value_counts().reset_index()
esp_depois.columns = ['Especialidade', 'Qtd_Depois']

esp_comparativo = pd.merge(esp_antes, esp_depois, on='Especialidade', how='outer').fillna(0)
esp_comparativo = esp_comparativo.sort_values(by=['Qtd_Antes','Qtd_Depois'], ascending=False)
print("\n📋 Especialidades (antes/depois) — tabela completa (ordenada):")
display(esp_comparativo.head(50))

# 8) Linha do tempo por paciente: contagem de eventos por MOMENTO e tipo
def tipo_atendimento(row):
    a = str(row['AGRUP_ASSISTENCIAL_G']).upper()
    e = str(row['ESPECIALIDADE']).upper()
    if 'EMERGENCIA' in a: return 'EMERGENCIA'
    if 'ELETIVA' in a and 'CARDIO' in e: return 'CARDIO_ELETIVA'
    if 'ELETIVA' in a: return 'ELETIVA_OUTRA'
    return 'OUTRO'

df_jornada_arritmia['TIPO_ATEND'] = df_jornada_arritmia.apply(tipo_atendimento, axis=1)

timeline = df_jornada_arritmia.groupby(['ID_PESSOA','MOMENTO','TIPO_ATEND']).size().unstack(fill_value=0).reset_index()
print("\n🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):")
display(timeline.head(50))

print("\n✅ Análise da jornada de ARRITMIA executada.")

🔍 Iniciando análise: jornada completa dos pacientes com ARRITMIA...
  - Padrão de busca: ARRITMIA CARDIACA NAO ESPECIFICADA|OUTRAS ARRITMIAS CARDÍACAS
🫀 IDs únicos com registro de ARRITMIA: 7
📂 Registros desses pacientes (toda jornada): 3,138
  - Registros 'Antes': 2243. Registros 'Depois': 833.
  - Métricas 'Antes' e 'Depois' calculadas.

📊 Resumo da Jornada (Arritmia):


,Período,Pacientes únicos (nesse período),Atendimentos emergência (total),Pacientes com emergência,Consultas cardio eletiva (total),Pacientes com cardio eletiva
0,Antes,7,44,5,32,7
1,Depois,7,6,3,24,6



📋 Especialidades (antes/depois) — tabela completa (ordenada):


,Especialidade,Qtd_Antes,Qtd_Depois
17,NAO INFORMADO,1718.0,586.0
9,FISIOTERAPIA,91.0,28.0
2,CARDIOLOGIA,90.0,62.0
5,CLINICA MEDICA,57.0,4.0
23,ORTOPEDIA E TRAUMATOLOGIA,49.0,5.0
30,RADIOLOGIA,46.0,17.0
22,OFTALMOLOGIA,43.0,42.0
0,ANESTESIOLOGIA,30.0,22.0
6,DERMATOLOGIA,15.0,5.0
25,PATOLOGIA,15.0,5.0



🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):


TIPO_ATEND,ID_PESSOA,MOMENTO,CARDIO_ELETIVA,ELETIVA_OUTRA,EMERGENCIA,OUTRO
0,72467,ANTES,2,12,6,274
1,72467,DEPOIS,2,1,1,136
2,72467,MESMA DATA,0,0,0,5
3,180733,ANTES,8,11,0,252
4,180733,DEPOIS,5,13,3,106
5,180733,MESMA DATA,0,0,0,2
6,398334,ANTES,6,16,3,160
7,398334,DEPOIS,3,8,2,106
8,398334,MESMA DATA,0,0,0,12
9,405426,ANTES,4,34,1,230



✅ Análise da jornada de ARRITMIA executada.


In [51]:
# =============================================================================
# Célula 5 (VERSÃO ARRITMIA): Análise 3 - Busca por procedimentos-chave
# =============================================================================
print("Iniciando Célula 5 (para Arritmia): Busca por procedimentos-chave (Angio/Cateter/Stent)...")

# 1. Palavras-chave dos procedimentos (são as mesmas)
keywords_procedimentos = [
    'ANGIOTOMOGRAFIA CORONARIANA',
    'CATETERISMO CARDIACO',
    'IMPLANTE DE STENT CORONARIO'
]

# 2. Padrão regex com OU
#    (Esta variável 'pattern' será reutilizada na Célula 6)
pattern = '|'.join(keywords_procedimentos)
print(f"  - Padrão de busca por procedimentos: {pattern}")

# 3. Filtrar registros com os procedimentos
#    --- ESTA É A MUDANÇA ---
#    Agora procuramos no DataFrame df_jornada_arritmia
try:
    df_jornada_arritmia['SERVICO'] = df_jornada_arritmia['SERVICO'].astype(str)
    mask = df_jornada_arritmia['SERVICO'].str.contains(pattern, case=False, na=False)
    df_proc = df_jornada_arritmia[mask].copy()
    print("  - Filtro de procedimentos aplicado ao df_jornada_arritmia.")
except Exception as e:
    print(f"ERRO: O DataFrame 'df_jornada_arritmia' não foi encontrado ou está sem a coluna 'SERVICO'. {e}")
    # Criar um DataFrame vazio para o código não quebrar
    df_proc = pd.DataFrame(columns=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'SERVICO', 'MOMENTO'])

# 4. REMOVER DUPLICATAS: mesmo paciente + mesma data + mesmo serviço = 1 evento
df_proc_unicos = df_proc.drop_duplicates(
    subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'SERVICO']
)

# 5. Selecionar apenas colunas relevantes
colunas_display = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'MOMENTO', 'SERVICO']

# 6. Resultados
if df_proc_unicos.empty:
    print("\nRESULTADO: Nenhum procedimento-chave encontrado para os pacientes com Arritmia.")
else:
    print(f"\nSUCESSO! Encontrados {df_proc_unicos.shape[0]} procedimentos-chave únicos.\n")

    print("-- Detalhe dos Procedimentos (1 por data/serviço) --")
    display(df_proc_unicos[colunas_display]
            .sort_values(by=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO']))

    print("\n-- Resumo por Paciente e Momento --")
    resumo = (df_proc_unicos
                .groupby(['ID_PESSOA', 'MOMENTO', 'SERVICO'])
                .size()
                .reset_index(name='Qtd_Eventos'))
    display(resumo)

print("\nAnálise 3 (Célula 5) concluída para Arritmia.")

Iniciando Célula 5 (para Arritmia): Busca por procedimentos-chave (Angio/Cateter/Stent)...
  - Padrão de busca por procedimentos: ANGIOTOMOGRAFIA CORONARIANA|CATETERISMO CARDIACO|IMPLANTE DE STENT CORONARIO
  - Filtro de procedimentos aplicado ao df_jornada_arritmia.

SUCESSO! Encontrados 10 procedimentos-chave únicos.

-- Detalhe dos Procedimentos (1 por data/serviço) --


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,MOMENTO,SERVICO
37345,72467,2025-03-27,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
122550,180733,2024-04-17,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
316139,398334,2024-04-01,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
323918,405426,2024-03-13,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
323601,405426,2024-04-12,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
323804,405426,2024-10-16,DEPOIS,IMPLANTE DE STENT CORONARIO COM OU SEM ANGIOPL...
511808,878822,2024-09-06,ANTES,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
512552,878822,2025-03-27,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
535198,972582,2023-07-03,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
552489,1021053,2024-08-19,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...



-- Resumo por Paciente e Momento --


,ID_PESSOA,MOMENTO,SERVICO,Qtd_Eventos
0,72467,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
1,180733,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
2,398334,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
3,405426,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
4,405426,DEPOIS,IMPLANTE DE STENT CORONARIO COM OU SEM ANGIOPL...,1
5,405426,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
6,878822,ANTES,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
7,878822,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
8,972582,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
9,1021053,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1



Análise 3 (Célula 5) concluída para Arritmia.


In [52]:
# =============================================================================
# Célula 6 (VERSÃO ARRITMIA): Análise 4 - Cronologia Real (Timeline em Dias)
# =============================================================================
import numpy as np
import pandas as pd

print("⏳ Iniciando Análise 4 (VERSÃO ARRITMIA): Construindo a Linha do Tempo Real (em dias)...")

# 0. Definir o DataFrame de base
df_base_arritmia = df_jornada_arritmia.copy()

# 1. Garantir que as colunas de data e tipo de atendimento existem e estão corretas
try:
    df_base_arritmia['DATA_ATENDIMENTO_FATO_PRO'] = pd.to_datetime(df_base_arritmia['DATA_ATENDIMENTO_FATO_PRO'])

    # Garantir que colunas de texto são strings
    for col in ['SERVICO', 'AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'DESCRICAO_CID']:
        if col in df_base_arritmia.columns:
            df_base_arritmia[col] = df_base_arritmia[col].astype(str)

    print("  - Coluna de datas convertida para formato datetime.")
    print("  - Colunas de texto (incluindo SERVICO) garantidas como string.")

except Exception as e:
    print(f"Erro na preparação inicial: {e}.")

# (Opcional) Recriar TIPO_ATEND por segurança
if 'TIPO_ATEND' not in df_base_arritmia.columns:
    print("  - Recriando coluna 'TIPO_ATEND'...")
    def tipo_atendimento(row):
        a = str(row['AGRUP_ASSISTENCIAL_G']).upper()
        e = str(row['ESPECIALIDADE']).upper()
        if 'EMERGENCIA' in a: return 'EMERGENCIA'
        if 'ELETIVA' in a and 'CARDIO' in e: return 'CARDIO_ELETIVA'
        if 'ELETIVA' in a: return 'ELETIVA_OUTRA'
        return 'OUTRO'
    df_base_arritmia['TIPO_ATEND'] = df_base_arritmia.apply(tipo_atendimento, axis=1)


# 2. Encontrar o "Dia Zero": a primeira data de registro de 'ARRITMIA' por paciente
#    (Verificar se 'pattern_arritmia' existe da Célula 4)
try:
    pattern_arritmia
except NameError:
    print("  - Recriando 'pattern_arritmia' (keywords de CID)...")
    keywords_arritmia = [
        'ARRITMIA CARDIACA NAO ESPECIFICADA',
        'OUTRAS ARRITMIAS CARDÍACAS'
    ]
    pattern_arritmia = '|'.join(keywords_arritmia)

mask_arritmia_temp = df_base_arritmia['DESCRICAO_CID'].str.contains(pattern_arritmia, case=False, na=False)
df_arritmia_eventos = df_base_arritmia[mask_arritmia_temp]

event_0_dates = df_arritmia_eventos.groupby('ID_PESSOA')['DATA_ATENDIMENTO_FATO_PRO'].min().reset_index()
event_0_dates.columns = ['ID_PESSOA', 'Data_Evento_0']

print("\n🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Arritmia):")
display(event_0_dates)

# 3. Juntar a 'Data_Evento_0' a todos os registros dos pacientes
df_jornada_timeline_arritmia = pd.merge(df_base_arritmia, event_0_dates, on='ID_PESSOA', how='left')

# 4. Calcular os dias relativos ao "Dia Zero"
df_jornada_timeline_arritmia['Dias_Desde_Evento_0'] = (df_jornada_timeline_arritmia['DATA_ATENDIMENTO_FATO_PRO'] - df_jornada_timeline_arritmia['Data_Evento_0']).dt.days

# 5. Definir as máscaras para TODOS os eventos-chave
#    (Verificar se 'pattern' de procedimentos existe da Célula 5)
try:
    pattern # Este é o 'pattern' dos procedimentos da Célula 5
except NameError:
    print("  - Recriando 'pattern' (keywords de procedimento)...")
    keywords_procedimentos = ['ANGIOTOMOGRAFIA CORONARIANA', 'CATETERISMO CARDIACO', 'IMPLANTE DE STENT CORONARIO']
    pattern = '|'.join(keywords_procedimentos)

print("  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline_arritmia)...")

# Todas as máscaras são definidas no df_jornada_timeline_arritmia
mask_proc = df_jornada_timeline_arritmia['SERVICO'].str.contains(pattern, case=False, na=False)
mask_emerg = df_jornada_timeline_arritmia['TIPO_ATEND'] == 'EMERGENCIA'
mask_cardio_elet = df_jornada_timeline_arritmia['TIPO_ATEND'] == 'CARDIO_ELETIVA'
# Recriamos a máscara de arritmia no DataFrame correto
mask_arritmia = df_jornada_timeline_arritmia['DESCRICAO_CID'].str.contains(pattern_arritmia, case=False, na=False)


# 6. Filtrar o DataFrame para conter APENAS os eventos-chave
mask_key_events = mask_arritmia | mask_proc | mask_emerg | mask_cardio_elet
df_timeline_filtrada = df_jornada_timeline_arritmia[mask_key_events].copy()

# 7. Criar Etiquetas claras para os eventos (em ordem de prioridade)
conditions = [
    df_timeline_filtrada['SERVICO'].str.contains('IMPLANTE DE STENT', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('CATETERISMO CARDIACO', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('ANGIOTOMOGRAFIA CORONARIANA', case=False, na=False),

    # --- A CORREÇÃO ESTÁ AQUI ---
    # Etiqueta "EVENTO" só é aplicada no "Dia Zero"
    (df_timeline_filtrada['DESCRICAO_CID'].str.contains(pattern_arritmia, case=False, na=False)) &
    (df_timeline_filtrada['DATA_ATENDIMENTO_FATO_PRO'] == df_timeline_filtrada['Data_Evento_0']),
    # -----------------------------

    df_timeline_filtrada['TIPO_ATEND'] == 'EMERGENCIA',
    df_timeline_filtrada['TIPO_ATEND'] == 'CARDIO_ELETIVA'
]

choices = [
    'PROC: Implante de Stent',
    'PROC: Cateterismo Cardiaco',
    'PROC: Angiotomografia COR',
    'EVENTO: Registro Arritmia', # Etiqueta atualizada
    'VISITA: Emergência',
    'VISITA: Cardio Eletiva'
]

df_timeline_filtrada['Evento_Label'] = np.select(conditions, choices, default='Outro (Reg. Arritmia Sec.)')

# 8. Preparar e exibir a Tabela-Resumo Final
colunas_finais = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Dias_Desde_Evento_0', 'Evento_Label']
df_timeline_final_view_arritmia = df_timeline_filtrada[colunas_finais]

# Remover os registros secundários (os 'Outros')
df_timeline_final_view_arritmia = df_timeline_final_view_arritmia[
    df_timeline_final_view_arritmia['Evento_Label'].str.contains('Outro') == False
].copy()

# Ordenar e remover duplicatas
df_timeline_final_view_arritmia = df_timeline_final_view_arritmia.sort_values(by=['ID_PESSOA', 'Dias_Desde_Evento_0'])
df_timeline_final_view_arritmia = df_timeline_final_view_arritmia.drop_duplicates(
    subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Evento_Label']
)

print("\n--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes com ARRITMIA) ---")
print("(Apenas o PRIMEIRO registro de Arritmia é mostrado como 'EVENTO')")
display(df_timeline_final_view_arritmia)

print("\n✅ Análise 4 (Célula 6) para ARRITMIA concluída. A cronologia está agora limpa.")

⏳ Iniciando Análise 4 (VERSÃO ARRITMIA): Construindo a Linha do Tempo Real (em dias)...
  - Coluna de datas convertida para formato datetime.
  - Colunas de texto (incluindo SERVICO) garantidas como string.

🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Arritmia):


,ID_PESSOA,Data_Evento_0
0,72467,2025-03-26
1,180733,2022-03-24
2,398334,2022-02-17
3,405426,2022-03-24
4,878822,2024-09-16
5,972582,2022-08-22
6,1021053,2024-02-27


  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline_arritmia)...

--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes com ARRITMIA) ---
(Apenas o PRIMEIRO registro de Arritmia é mostrado como 'EVENTO')


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,Dias_Desde_Evento_0,Evento_Label
130,72467,2021-12-12,-1200,VISITA: Emergência
131,72467,2022-01-22,-1159,VISITA: Emergência
132,72467,2023-03-31,-726,VISITA: Emergência
121,72467,2023-04-14,-712,VISITA: Cardio Eletiva
133,72467,2023-05-06,-690,VISITA: Emergência
...,...,...,...,...
3004,1021053,2024-04-08,41,VISITA: Cardio Eletiva
3007,1021053,2024-08-05,160,VISITA: Cardio Eletiva
3100,1021053,2024-08-19,174,PROC: Angiotomografia COR
3009,1021053,2024-10-07,223,VISITA: Cardio Eletiva



✅ Análise 4 (Célula 6) para ARRITMIA concluída. A cronologia está agora limpa.


In [53]:
# =============================================================================
# Célula 7 (VERSÃO ARRITMIA): Visualização da Cronologia (Timeline Plot)
# =============================================================================
import plotly.express as px

print("📊 Iniciando Célula 7 (VERSÃO ARRITMIA): Geração do Gráfico de Cronologia...")

# --- Preparação para o Gráfico ---
# 1. Usar o DataFrame final da ARRITMIA
df_plot = df_timeline_final_view_arritmia.copy()

# 2. Garantir que o ID do Paciente é tratado como texto (categoria)
df_plot['ID_PESSOA'] = df_plot['ID_PESSOA'].astype(str)

# --- Calcular o Range Completo do Eixo X ---
# Vamos encontrar o dia mínimo (ex: -1200) e o máximo
min_dias = df_plot['Dias_Desde_Evento_0'].min()
max_dias = df_plot['Dias_Desde_Evento_0'].max()

# Adicionar uma "margem" visual (ex: 30 dias)
min_dias_com_margem = min_dias - 30
max_dias_com_margem = max_dias + 30

print(f"  - Dados preparados. Range do eixo X definido de {min_dias_com_margem} até {max_dias_com_margem} dias.")

# --- Criação do Gráfico Interativo ---
fig = px.scatter(
    data_frame=df_plot,
    x='Dias_Desde_Evento_0',
    y='ID_PESSOA',
    color='Evento_Label',                  # Define as cores com base na etiqueta do evento
    hover_data=[                         # O que mostrar ao passar o rato
        'DATA_ATENDIMENTO_FATO_PRO',
        'Evento_Label',
        'Dias_Desde_Evento_0'
    ],
    labels={                             # Renomear os eixos para ficarem claros
        'Dias_Desde_Evento_0': 'Dias desde o Evento Zero (1ª Arritmia)',
        'ID_PESSOA': 'Paciente',
        'Evento_Label': 'Tipo de Evento'
    },
    # Título atualizado
    title='Cronologia da Jornada do Paciente (Dias desde a 1ª Arritmia)'
)

# --- Melhorias no Gráfico ---
# 1. Adicionar uma linha vertical vermelha no "Dia 0"
fig.add_vline(
    x=0,
    line_width=2,
    line_dash="dash",
    line_color="red",
    annotation_text="Dia 0 (Arritmia)", # Texto atualizado
    annotation_position="bottom right"
)

# 2. Melhorar a visualização para 7 pacientes
#    (Reativamos as etiquetas no eixo Y, pois são poucos pacientes)
fig.update_yaxes(
    type='category',
    showticklabels=True,
    title='Pacientes'
)

# 3. Ordenar o eixo Y (pacientes) E APLICAR O NOVO RANGE DO EIXO X
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_range=[min_dias_com_margem, max_dias_com_margem],
    height=500 # Podemos usar uma altura menor, pois são poucos pacientes
)

print("  - Gráfico gerado com sucesso.")

# --- Exibir o Gráfico ---
fig.show()

print("\n✅ Visualização (Arritmia) concluída.")

📊 Iniciando Célula 7 (VERSÃO ARRITMIA): Geração do Gráfico de Cronologia...
  - Dados preparados. Range do eixo X definido de -1230 até 1217 dias.
  - Gráfico gerado com sucesso.



✅ Visualização (Arritmia) concluída.


In [54]:
# =============================================================================
# Célula 9 (VERSÃO ARRITMIA): Análise 6 - Cálculo de Métricas Preventivas
# =============================================================================

print("🚨 Iniciando Análise 6 (VERSÃO ARRITMIA): Calculando KPIs Preventivos (Sinais de Alerta Pré-Arritmia)...")

# 1. Criar um DataFrame focado APENAS em eventos ANTES do Dia 0
#    --- ESTA É A MUDANÇA ---
df_pre_arritmia = df_timeline_final_view_arritmia[
    df_timeline_final_view_arritmia['Dias_Desde_Evento_0'] < 0
].copy()

if df_pre_arritmia.empty:
    print("  - Nenhum evento registrado antes do 'Dia Zero' para os pacientes com Arritmia.")

else:
    print(f"  - {df_pre_arritmia.shape[0]} eventos pré-arritmia encontrados para análise.")


# --- 2. Calcular os KPIs Preventivos ---
# (Aplicando a lógica ao df_pre_arritmia)

# KPI 1: Qtd. Visitas Cardio Eletiva (Pré-Arritmia)
df_cardio_pre = df_pre_arritmia[
    df_pre_arritmia['Evento_Label'] == 'VISITA: Cardio Eletiva'
]
kpi_cardio_pre = df_cardio_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Cardio_Eletiva_Pre_Arritmia')

# KPI 2: Qtd. Visitas Emergência (Pré-Arritmia)
df_emerg_pre = df_pre_arritmia[
    df_pre_arritmia['Evento_Label'] == 'VISITA: Emergência'
]
kpi_emerg_pre = df_emerg_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Emergencia_Pre_Arritmia')

# KPI 3: Qtd. Procedimentos Chave (Pré-Arritmia) - (Angio, Cateter, Stent)
df_proc_pre = df_pre_arritmia[
    df_pre_arritmia['Evento_Label'].str.contains('PROC:')
]
kpi_proc_pre = df_proc_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Procedimentos_Pre_Arritmia')

print("  - KPIs preventivos calculados.")

# --- 3. Consolidar a tabela de KPIs Preventivos ---

# Começamos com a lista completa dos 7 pacientes de Arritmia
df_pacientes_arritmia = df_timeline_final_view_arritmia[['ID_PESSOA']].drop_duplicates()

# Juntar os KPIs preventivos
df_kpi_preventivo_arritmia = df_pacientes_arritmia.merge(kpi_cardio_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo_arritmia = df_kpi_preventivo_arritmia.merge(kpi_emerg_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo_arritmia = df_kpi_preventivo_arritmia.merge(kpi_proc_pre, on='ID_PESSOA', how='left')

# Substituir 'NaN' (Nulo) por 0. 'NaN' aqui significa "nenhum evento", ou seja, 0.
cols_kpi = ['Qtd_Cardio_Eletiva_Pre_Arritmia', 'Qtd_Emergencia_Pre_Arritmia', 'Qtd_Procedimentos_Pre_Arritmia']
df_kpi_preventivo_arritmia[cols_kpi] = df_kpi_preventivo_arritmia[cols_kpi].fillna(0).astype(int)

print("  - KPIs preventivos consolidados.")

# --- 4. Exibir Resultados ---
print("\n--- 🚨 Métricas Preventivas (Comportamento Pré-Arritmia) ---")
print("(Contagem de eventos ANTES do 'Dia Zero' de cada paciente)")
display(df_kpi_preventivo_arritmia)

# --- 5. Calcular Médias Gerais ---
print("\n--- 📊 Resumo de Métricas Preventivas (Médias) ---")
media_cardio = df_kpi_preventivo_arritmia['Qtd_Cardio_Eletiva_Pre_Arritmia'].mean()
media_emerg = df_kpi_preventivo_arritmia['Qtd_Emergencia_Pre_Arritmia'].mean()
media_proc = df_kpi_preventivo_arritmia['Qtd_Procedimentos_Pre_Arritmia'].mean()

print(f"  - Média de Consultas Cardio Eletivas (Pré-Arritmia): {media_cardio:.2f} por paciente")
print(f"  - Média de Visitas à Emergência (Pré-Arritmia): {media_emerg:.2f} por paciente")
print(f"  - Média de Procedimentos-Chave (Pré-Arritmia): {media_proc:.2f} por paciente")


print("\n✅ Análise 6 (KPIs Preventivos para Arritmia) concluída.")

🚨 Iniciando Análise 6 (VERSÃO ARRITMIA): Calculando KPIs Preventivos (Sinais de Alerta Pré-Arritmia)...
  - 38 eventos pré-arritmia encontrados para análise.
  - KPIs preventivos calculados.
  - KPIs preventivos consolidados.

--- 🚨 Métricas Preventivas (Comportamento Pré-Arritmia) ---
(Contagem de eventos ANTES do 'Dia Zero' de cada paciente)


,ID_PESSOA,Qtd_Cardio_Eletiva_Pre_Arritmia,Qtd_Emergencia_Pre_Arritmia,Qtd_Procedimentos_Pre_Arritmia
0,72467,1,5,0
1,180733,1,0,0
2,398334,0,1,0
3,405426,1,0,0
4,878822,1,23,1
5,972582,2,0,0
6,1021053,2,0,0



--- 📊 Resumo de Métricas Preventivas (Médias) ---
  - Média de Consultas Cardio Eletivas (Pré-Arritmia): 1.14 por paciente
  - Média de Visitas à Emergência (Pré-Arritmia): 4.14 por paciente
  - Média de Procedimentos-Chave (Pré-Arritmia): 0.14 por paciente

✅ Análise 6 (KPIs Preventivos para Arritmia) concluída.


In [55]:
# =============================================================================
# Célula 4 (VERSÃO INSUFICIÊNCIA): Jornada completa dos pacientes
# =============================================================================
import pandas as pd
import numpy as np

print("🔍 Iniciando análise: jornada completa dos pacientes com INSUFICIÊNCIA (Cardíaca/Hipertensiva/Valvar)...")

# 1) identificar IDs que em algum momento tiveram um dos CIDs (qualquer linha)
# --- NOVAS PALAVRAS-CHAVE ---
keywords_insuficiencia = [
    'INSUFICIENCIA AORTICA',
    'INSUFICIENCIA CARDIACA NAO ESPECIFICADA',
    'DOENÇA CARDIACA HIPERTENSIVA SEM INSUFICIENCIA',
    'INSUFICIENCIA CARDIACA CONGESTIVA',
    'DOENÇA CARDIACA HIPERTENSIVA COM INSUFICIENCIA',
    'INSUFICIENCIA MITRAL'
]
pattern_insuficiencia = '|'.join(keywords_insuficiencia)
print(f" ---------------------------------------------------------------------------------")
print(f"  - Padrão de busca: {pattern_insuficiencia}")
print(f" ---------------------------------------------------------------------------------")


mask_insuficiencia_any = df['DESCRICAO_CID'].str.contains(pattern_insuficiencia, case=False, na=False)
ids_insuficiencia = df.loc[mask_insuficiencia_any, 'ID_PESSOA'].unique()
print(f"🫀 IDs únicos com registro de INSUFICIÊNCIA (Cardíaca/Hipertensiva/Valvar): {len(ids_insuficiencia)}")

# 2) obter todas as linhas desses pacientes (toda a jornada)
#    (Renomeado para df_jornada_insuficiencia)
df_jornada_insuficiencia = df[df['ID_PESSOA'].isin(ids_insuficiencia)].copy()
print(f"📂 Registros desses pacientes (toda jornada): {df_jornada_insuficiencia.shape[0]:,}")

# 3) garantir colunas de interesse como texto
for c in ['AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'MOMENTO', 'DESCRICAO_CID']:
    df_jornada_insuficiencia[c] = df_jornada_insuficiencia[c].astype(str)

# 4) separar antes/depois pelo campo MOMENTO
antes = df_jornada_insuficiencia[df_jornada_insuficiencia['MOMENTO'].str.upper().str.strip() == 'ANTES']
depois = df_jornada_insuficiencia[df_jornada_insuficiencia['MOMENTO'].str.upper().str.strip() == 'DEPOIS']
print(f"  - Registros 'Antes': {antes.shape[0]}. Registros 'Depois': {depois.shape[0]}.")

# 5) métricas desejadas:

# --- Métricas do período ANTES ---
mask_emerg_antes = antes['AGRUP_ASSISTENCIAL_G'].str.contains('EMERGENCIA', case=False, na=False)
mask_eletiva_antes = antes['AGRUP_ASSISTENCIAL_G'].str.contains('ELETIVA', case=False, na=False)
mask_cardio_antes = antes['ESPECIALIDADE'].str.contains('CARDIO', case=False, na=False)

emerg_antes_total = antes[mask_emerg_antes].shape[0]
emerg_antes_pacientes = antes[mask_emerg_antes]['ID_PESSOA'].nunique()
cardio_elet_antes_total = antes[mask_eletiva_antes & mask_cardio_antes].shape[0]
cardio_elet_antes_pacientes = antes[mask_eletiva_antes & mask_cardio_antes]['ID_PESSOA'].nunique()

# --- Métricas do período DEPOIS ---
mask_emerg_depois = depois['AGRUP_ASSISTENCIAL_G'].str.contains('EMERGENCIA', case=False, na=False)
mask_eletiva_depois = depois['AGRUP_ASSISTENCIAL_G'].str.contains('ELETIVA', case=False, na=False)
mask_cardio_depois = depois['ESPECIALIDADE'].str.contains('CARDIO', case=False, na=False)

emerg_depois_total = depois[mask_emerg_depois].shape[0]
emerg_depois_pacientes = depois[mask_emerg_depois]['ID_PESSOA'].nunique()
cardio_elet_depois_total = depois[mask_eletiva_depois & mask_cardio_depois].shape[0]
cardio_elet_depois_pacientes = depois[mask_eletiva_depois & mask_cardio_depois]['ID_PESSOA'].nunique()
print("  - Métricas 'Antes' e 'Depois' calculadas.")

# 6) montar resumo
resumo = pd.DataFrame({
    'Período': ['Antes', 'Depois'],
    'Pacientes únicos (nesse período)': [antes['ID_PESSOA'].nunique(), depois['ID_PESSOA'].nunique()],
    'Atendimentos emergência (total)': [emerg_antes_total, emerg_depois_total],
    'Pacientes com emergência': [emerg_antes_pacientes, emerg_depois_pacientes],
    'Consultas cardio eletiva (total)': [cardio_elet_antes_total, cardio_elet_depois_total],
    'Pacientes com cardio eletiva': [cardio_elet_antes_pacientes, cardio_elet_depois_pacientes]
})
print("\n📊 Resumo da Jornada (Insuficiência):")
display(resumo)

# 7) Top especialidades (antes / depois)
esp_antes = antes['ESPECIALIDADE'].value_counts().reset_index()
esp_antes.columns = ['Especialidade', 'Qtd_Antes']

esp_depois = depois['ESPECIALIDADE'].value_counts().reset_index()
esp_depois.columns = ['Especialidade', 'Qtd_Depois']

esp_comparativo = pd.merge(esp_antes, esp_depois, on='Especialidade', how='outer').fillna(0)
esp_comparativo = esp_comparativo.sort_values(by=['Qtd_Antes','Qtd_Depois'], ascending=False)
print("\n📋 Especialidades (antes/depois) — tabela completa (ordenada):")
display(esp_comparativo.head(50))

# 8) Linha do tempo por paciente: contagem de eventos por MOMENTO e tipo
def tipo_atendimento(row):
    a = str(row['AGRUP_ASSISTENCIAL_G']).upper()
    e = str(row['ESPECIALIDADE']).upper()
    if 'EMERGENCIA' in a: return 'EMERGENCIA'
    if 'ELETIVA' in a and 'CARDIO' in e: return 'CARDIO_ELETIVA'
    if 'ELETIVA' in a: return 'ELETIVA_OUTRA'
    return 'OUTRO'

df_jornada_insuficiencia['TIPO_ATEND'] = df_jornada_insuficiencia.apply(tipo_atendimento, axis=1)

timeline = df_jornada_insuficiencia.groupby(['ID_PESSOA','MOMENTO','TIPO_ATEND']).size().unstack(fill_value=0).reset_index()
print("\n🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):")
display(timeline.head(50))

print("\n✅ Análise da jornada de INSUFICIÊNCIA executada.")

🔍 Iniciando análise: jornada completa dos pacientes com INSUFICIÊNCIA (Cardíaca/Hipertensiva/Valvar)...
 ---------------------------------------------------------------------------------
  - Padrão de busca: INSUFICIENCIA AORTICA|INSUFICIENCIA CARDIACA NAO ESPECIFICADA|DOENÇA CARDIACA HIPERTENSIVA SEM INSUFICIENCIA|INSUFICIENCIA CARDIACA CONGESTIVA|DOENÇA CARDIACA HIPERTENSIVA COM INSUFICIENCIA|INSUFICIENCIA MITRAL
 ---------------------------------------------------------------------------------
🫀 IDs únicos com registro de INSUFICIÊNCIA (Cardíaca/Hipertensiva/Valvar): 6
📂 Registros desses pacientes (toda jornada): 2,554
  - Registros 'Antes': 1375. Registros 'Depois': 1144.
  - Métricas 'Antes' e 'Depois' calculadas.

📊 Resumo da Jornada (Insuficiência):


,Período,Pacientes únicos (nesse período),Atendimentos emergência (total),Pacientes com emergência,Consultas cardio eletiva (total),Pacientes com cardio eletiva
0,Antes,6,13,5,23,5
1,Depois,6,10,3,23,5



📋 Especialidades (antes/depois) — tabela completa (ordenada):


,Especialidade,Qtd_Antes,Qtd_Depois
22,NAO INFORMADO,799.0,555.0
13,FISIOTERAPIA,157.0,234.0
29,PATOLOGIA,136.0,136.0
9,CLINICA MEDICA,78.0,75.0
3,CARDIOLOGIA,59.0,51.0
25,OFTALMOLOGIA,24.0,23.0
32,RADIOLOGIA,24.0,16.0
27,ORTOPEDIA E TRAUMATOLOGIA,21.0,3.0
19,MEDICINA INTENSIVA,11.0,12.0
16,GINECOLOGIA E OBSTETRICIA,9.0,3.0



🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):


TIPO_ATEND,ID_PESSOA,MOMENTO,CARDIO_ELETIVA,ELETIVA_OUTRA,EMERGENCIA,OUTRO
0,133556,ANTES,5,10,3,330
1,133556,DEPOIS,11,5,2,267
2,133556,MESMA DATA,0,0,0,1
3,196323,ANTES,10,19,4,324
4,196323,DEPOIS,6,5,0,157
5,196323,MESMA DATA,0,0,0,1
6,204372,ANTES,1,3,2,163
7,204372,DEPOIS,4,8,6,161
8,204372,MESMA DATA,0,0,0,1
9,212498,ANTES,0,3,2,248



✅ Análise da jornada de INSUFICIÊNCIA executada.


In [56]:
# =============================================================================
# Célula 5 (VERSÃO INSUFICIÊNCIA): Análise 3 - Busca por procedimentos-chave
# =============================================================================
print("Iniciando Célula 5 (para Insuficiência): Busca por procedimentos-chave (Angio/Cateter/Stent)...")

# 1. Palavras-chave dos procedimentos (são as mesmas)
keywords_procedimentos = [
    'ANGIOTOMOGRAFIA CORONARIANA',
    'CATETERISMO CARDIACO',
    'IMPLANTE DE STENT CORONARIO'
]

# 2. Padrão regex com OU
#    (Esta variável 'pattern' será reutilizada na Célula 6)
pattern = '|'.join(keywords_procedimentos)
print(f"  - Padrão de busca por procedimentos: {pattern}")

# 3. Filtrar registros com os procedimentos
#    --- ESTA É A MUDANÇA ---
#    Agora procuramos no DataFrame df_jornada_insuficiencia
try:
    df_jornada_insuficiencia['SERVICO'] = df_jornada_insuficiencia['SERVICO'].astype(str)
    mask = df_jornada_insuficiencia['SERVICO'].str.contains(pattern, case=False, na=False)
    df_proc = df_jornada_insuficiencia[mask].copy()
    print("  - Filtro de procedimentos aplicado ao df_jornada_insuficiencia.")
except Exception as e:
    print(f"ERRO: O DataFrame 'df_jornada_insuficiencia' não foi encontrado ou está sem a coluna 'SERVICO'. {e}")
    # Criar um DataFrame vazio para o código não quebrar
    df_proc = pd.DataFrame(columns=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'SERVICO', 'MOMENTO'])

# 4. REMOVER DUPLICATAS: mesmo paciente + mesma data + mesmo serviço = 1 evento
df_proc_unicos = df_proc.drop_duplicates(
    subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'SERVICO']
)

# 5. Selecionar apenas colunas relevantes
colunas_display = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'MOMENTO', 'SERVICO']

# 6. Resultados
if df_proc_unicos.empty:
    print("\nRESULTADO: Nenhum procedimento-chave encontrado para os pacientes com Insuficiência.")
else:
    print(f"\nSUCESSO! Encontrados {df_proc_unicos.shape[0]} procedimentos-chave únicos.\n")

    print("-- Detalhe dos Procedimentos (1 por data/serviço) --")
    display(df_proc_unicos[colunas_display]
            .sort_values(by=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO']))

    print("\n-- Resumo por Paciente e Momento --")
    resumo = (df_proc_unicos
                .groupby(['ID_PESSOA', 'MOMENTO', 'SERVICO'])
                .size()
                .reset_index(name='Qtd_Eventos'))
    display(resumo)

print("\nAnálise 3 (Célula 5) concluída para Insuficiência.")

Iniciando Célula 5 (para Insuficiência): Busca por procedimentos-chave (Angio/Cateter/Stent)...
  - Padrão de busca por procedimentos: ANGIOTOMOGRAFIA CORONARIANA|CATETERISMO CARDIACO|IMPLANTE DE STENT CORONARIO
  - Filtro de procedimentos aplicado ao df_jornada_insuficiencia.

SUCESSO! Encontrados 11 procedimentos-chave únicos.

-- Detalhe dos Procedimentos (1 por data/serviço) --


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,MOMENTO,SERVICO
90821,133556,2023-06-16,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
90408,133556,2023-07-11,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
130601,196323,2024-10-31,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
130217,196323,2024-12-18,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
130408,196323,2025-01-23,DEPOIS,IMPLANTE DE STENT CORONARIO COM OU SEM ANGIOPL...
136299,204372,2023-06-22,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
136220,204372,2023-11-08,DEPOIS,PACOTE-SISPAC - CATETERISMO CARDIACO E E/OU D ...
136221,204372,2025-05-09,DEPOIS,PACOTE-SISPAC - CATETERISMO CARDIACO E E/OU D ...
141054,212498,2023-12-08,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
276656,357598,2023-10-05,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...



-- Resumo por Paciente e Momento --


,ID_PESSOA,MOMENTO,SERVICO,Qtd_Eventos
0,133556,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
1,133556,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
2,196323,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
3,196323,DEPOIS,IMPLANTE DE STENT CORONARIO COM OU SEM ANGIOPL...,1
4,196323,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
5,204372,DEPOIS,PACOTE-SISPAC - CATETERISMO CARDIACO E E/OU D ...,2
6,204372,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
7,212498,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
8,357598,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
9,374899,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1



Análise 3 (Célula 5) concluída para Insuficiência.


In [57]:
# =============================================================================
# Célula 6 (VERSÃO INSUFICIÊNCIA): Análise 4 - Cronologia Real (Timeline em Dias)
# =============================================================================
import numpy as np
import pandas as pd

print("⏳ Iniciando Análise 4 (VERSÃO INSUFICIÊNCIA): Construindo a Linha do Tempo Real (em dias)...")

# 0. Definir o DataFrame de base
df_base_insuficiencia = df_jornada_insuficiencia.copy()

# 1. Garantir que as colunas de data e tipo de atendimento existem e estão corretas
try:
    df_base_insuficiencia['DATA_ATENDIMENTO_FATO_PRO'] = pd.to_datetime(df_base_insuficiencia['DATA_ATENDIMENTO_FATO_PRO'])

    # Garantir que colunas de texto são strings
    for col in ['SERVICO', 'AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'DESCRICAO_CID']:
        if col in df_base_insuficiencia.columns:
            df_base_insuficiencia[col] = df_base_insuficiencia[col].astype(str)

    print("  - Coluna de datas convertida para formato datetime.")
    print("  - Colunas de texto (incluindo SERVICO) garantidas como string.")

except Exception as e:
    print(f"Erro na preparação inicial: {e}.")

# (Opcional) Recriar TIPO_ATEND por segurança
if 'TIPO_ATEND' not in df_base_insuficiencia.columns:
    print("  - Recriando coluna 'TIPO_ATEND'...")
    def tipo_atendimento(row):
        a = str(row['AGRUP_ASSISTENCIAL_G']).upper()
        e = str(row['ESPECIALIDADE']).upper()
        if 'EMERGENCIA' in a: return 'EMERGENCIA'
        if 'ELETIVA' in a and 'CARDIO' in e: return 'CARDIO_ELETIVA'
        if 'ELETIVA' in a: return 'ELETIVA_OUTRA'
        return 'OUTRO'
    df_base_insuficiencia['TIPO_ATEND'] = df_base_insuficiencia.apply(tipo_atendimento, axis=1)


# 2. Encontrar o "Dia Zero": a primeira data de registro de 'INSUFICIÊNCIA' por paciente
#    (Verificar se 'pattern_insuficiencia' existe da Célula 4)
try:
    pattern_insuficiencia
except NameError:
    print("  - Recriando 'pattern_insuficiencia' (keywords de CID)...")
    keywords_insuficiencia = [
        'INSUFICIENCIA AORTICA',
        'INSUFICIENCIA CARDIACA NAO ESPECIFICADA',
        'DOENÇA CARDIACA HIPERTENSIVA SEM INSUFICIENCIA',
        'INSUFICIENCIA CARDIACA CONGESTIVA',
        'DOENÇA CARDIACA HIPERTENSIVA COM INSUFICIENCIA',
        'INSUFICIENCIA MITRAL'
    ]
    pattern_insuficiencia = '|'.join(keywords_insuficiencia)

mask_insuficiencia_temp = df_base_insuficiencia['DESCRICAO_CID'].str.contains(pattern_insuficiencia, case=False, na=False)
df_insuficiencia_eventos = df_base_insuficiencia[mask_insuficiencia_temp]

event_0_dates = df_insuficiencia_eventos.groupby('ID_PESSOA')['DATA_ATENDIMENTO_FATO_PRO'].min().reset_index()
event_0_dates.columns = ['ID_PESSOA', 'Data_Evento_0']

print("\n🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Insuficiência):")
display(event_0_dates)

# 3. Juntar a 'Data_Evento_0' a todos os registros dos pacientes
df_jornada_timeline_insuficiencia = pd.merge(df_base_insuficiencia, event_0_dates, on='ID_PESSOA', how='left')

# 4. Calcular os dias relativos ao "Dia Zero"
df_jornada_timeline_insuficiencia['Dias_Desde_Evento_0'] = (df_jornada_timeline_insuficiencia['DATA_ATENDIMENTO_FATO_PRO'] - df_jornada_timeline_insuficiencia['Data_Evento_0']).dt.days

# 5. Definir as máscaras para TODOS os eventos-chave
#    (Verificar se 'pattern' de procedimentos existe da Célula 5)
try:
    pattern # Este é o 'pattern' dos procedimentos da Célula 5
except NameError:
    print("  - Recriando 'pattern' (keywords de procedimento)...")
    keywords_procedimentos = ['ANGIOTOMOGRAFIA CORONARIANA', 'CATETERISMO CARDIACO', 'IMPLANTE DE STENT CORONARIO']
    pattern = '|'.join(keywords_procedimentos)

print("  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline_insuficiencia)...")

# Todas as máscaras são definidas no df_jornada_timeline_insuficiencia
mask_proc = df_jornada_timeline_insuficiencia['SERVICO'].str.contains(pattern, case=False, na=False)
mask_emerg = df_jornada_timeline_insuficiencia['TIPO_ATEND'] == 'EMERGENCIA'
mask_cardio_elet = df_jornada_timeline_insuficiencia['TIPO_ATEND'] == 'CARDIO_ELETIVA'
# Recriamos a máscara de insuficiencia no DataFrame correto
mask_insuficiencia = df_jornada_timeline_insuficiencia['DESCRICAO_CID'].str.contains(pattern_insuficiencia, case=False, na=False)


# 6. Filtrar o DataFrame para conter APENAS os eventos-chave
mask_key_events = mask_insuficiencia | mask_proc | mask_emerg | mask_cardio_elet
df_timeline_filtrada = df_jornada_timeline_insuficiencia[mask_key_events].copy()

# 7. Criar Etiquetas claras para os eventos (em ordem de prioridade)
conditions = [
    df_timeline_filtrada['SERVICO'].str.contains('IMPLANTE DE STENT', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('CATETERISMO CARDIACO', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('ANGIOTOMOGRAFIA CORONARIANA', case=False, na=False),

    # --- A CORREÇÃO ESTÁ AQUI ---
    # Etiqueta "EVENTO" só é aplicada no "Dia Zero"
    (df_timeline_filtrada['DESCRICAO_CID'].str.contains(pattern_insuficiencia, case=False, na=False)) &
    (df_timeline_filtrada['DATA_ATENDIMENTO_FATO_PRO'] == df_timeline_filtrada['Data_Evento_0']),
    # -----------------------------

    df_timeline_filtrada['TIPO_ATEND'] == 'EMERGENCIA',
    df_timeline_filtrada['TIPO_ATEND'] == 'CARDIO_ELETIVA'
]

choices = [
    'PROC: Implante de Stent',
    'PROC: Cateterismo Cardiaco',
    'PROC: Angiotomografia COR',
    'EVENTO: Registro Insuficiência', # Etiqueta atualizada
    'VISITA: Emergência',
    'VISITA: Cardio Eletiva'
]

df_timeline_filtrada['Evento_Label'] = np.select(conditions, choices, default='Outro (Reg. Insuf. Sec.)')

# 8. Preparar e exibir a Tabela-Resumo Final
colunas_finais = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Dias_Desde_Evento_0', 'Evento_Label']
df_timeline_final_view_insuficiencia = df_timeline_filtrada[colunas_finais]

# Remover os registros secundários (os 'Outros')
df_timeline_final_view_insuficiencia = df_timeline_final_view_insuficiencia[
    df_timeline_final_view_insuficiencia['Evento_Label'].str.contains('Outro') == False
].copy()

# Ordenar e remover duplicatas
df_timeline_final_view_insuficiencia = df_timeline_final_view_insuficiencia.sort_values(by=['ID_PESSOA', 'Dias_Desde_Evento_0'])
df_timeline_final_view_insuficiencia = df_timeline_final_view_insuficiencia.drop_duplicates(
    subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Evento_Label']
)

print("\n--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes com INSUFICIÊNCIA) ---")
print("(Apenas o PRIMEIRO registro de Insuficiência é mostrado como 'EVENTO')")
display(df_timeline_final_view_insuficiencia)

print("\n✅ Análise 4 (Célula 6) para INSUFICIÊNCIA concluída. A cronologia está agora limpa.")

⏳ Iniciando Análise 4 (VERSÃO INSUFICIÊNCIA): Construindo a Linha do Tempo Real (em dias)...
  - Coluna de datas convertida para formato datetime.
  - Colunas de texto (incluindo SERVICO) garantidas como string.

🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Insuficiência):


,ID_PESSOA,Data_Evento_0
0,133556,2023-04-11
1,196323,2022-11-17
2,204372,2023-01-20
3,212498,2023-12-07
4,357598,2022-01-13
5,374899,2022-02-17


  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline_insuficiencia)...

--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes com INSUFICIÊNCIA) ---
(Apenas o PRIMEIRO registro de Insuficiência é mostrado como 'EVENTO')


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,Dias_Desde_Evento_0,Evento_Label
179,133556,2021-12-31,-466,VISITA: Emergência
151,133556,2022-07-15,-270,VISITA: Cardio Eletiva
155,133556,2022-11-17,-145,VISITA: Cardio Eletiva
180,133556,2023-03-22,-20,VISITA: Emergência
7,133556,2023-04-11,0,EVENTO: Registro Insuficiência
...,...,...,...,...
2434,374899,2023-01-20,337,VISITA: Cardio Eletiva
2436,374899,2023-05-13,450,VISITA: Cardio Eletiva
2438,374899,2024-01-11,693,VISITA: Cardio Eletiva
2445,374899,2025-01-24,1072,VISITA: Cardio Eletiva



✅ Análise 4 (Célula 6) para INSUFICIÊNCIA concluída. A cronologia está agora limpa.


In [58]:
# =============================================================================
# Célula 7 (VERSÃO INSUFICIÊNCIA): Visualização da Cronologia (Timeline Plot)
# =============================================================================
import plotly.express as px

print("📊 Iniciando Célula 7 (VERSÃO INSUFICIÊNCIA): Geração do Gráfico de Cronologia...")

# --- Preparação para o Gráfico ---
# 1. Usar o DataFrame final da INSUFICIÊNCIA
df_plot = df_timeline_final_view_insuficiencia.copy()

# 2. Garantir que o ID do Paciente é tratado como texto (categoria)
df_plot['ID_PESSOA'] = df_plot['ID_PESSOA'].astype(str)

# --- Calcular o Range Completo do Eixo X ---
# Vamos encontrar o dia mínimo (ex: -466) e o máximo
min_dias = df_plot['Dias_Desde_Evento_0'].min()
max_dias = df_plot['Dias_Desde_Evento_0'].max()

# Adicionar uma "margem" visual (ex: 30 dias)
min_dias_com_margem = min_dias - 30
max_dias_com_margem = max_dias + 30

print(f"  - Dados preparados. Range do eixo X definido de {min_dias_com_margem} até {max_dias_com_margem} dias.")

# --- Criação do Gráfico Interativo ---
fig = px.scatter(
    data_frame=df_plot,
    x='Dias_Desde_Evento_0',
    y='ID_PESSOA',
    color='Evento_Label',                  # Define as cores com base na etiqueta do evento
    hover_data=[                         # O que mostrar ao passar o rato
        'DATA_ATENDIMENTO_FATO_PRO',
        'Evento_Label',
        'Dias_Desde_Evento_0'
    ],
    labels={                             # Renomear os eixos para ficarem claros
        'Dias_Desde_Evento_0': 'Dias desde o Evento Zero (1ª Insuficiência)',
        'ID_PESSOA': 'Paciente',
        'Evento_Label': 'Tipo de Evento'
    },
    # Título atualizado
    title='Cronologia da Jornada do Paciente (Dias desde a 1ª Insuficiência)'
)

# --- Melhorias no Gráfico ---
# 1. Adicionar uma linha vertical vermelha no "Dia 0"
fig.add_vline(
    x=0,
    line_width=2,
    line_dash="dash",
    line_color="red",
    annotation_text="Dia 0 (Insuficiência)", # Texto atualizado
    annotation_position="bottom right"
)

# 2. Melhorar a visualização para 6 pacientes
#    (Etiquetas ativadas no eixo Y, pois são poucos pacientes)
fig.update_yaxes(
    type='category',
    showticklabels=True,
    title='Pacientes'
)

# 3. Ordenar o eixo Y (pacientes) E APLICAR O NOVO RANGE DO EIXO X
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_range=[min_dias_com_margem, max_dias_com_margem],
    height=500 # Altura de 500px é boa para 6 pacientes
)

print("  - Gráfico gerado com sucesso.")

# --- Exibir o Gráfico ---
fig.show()

print("\n✅ Visualização (Insuficiência) concluída.")

📊 Iniciando Célula 7 (VERSÃO INSUFICIÊNCIA): Geração do Gráfico de Cronologia...
  - Dados preparados. Range do eixo X definido de -496 até 1290 dias.
  - Gráfico gerado com sucesso.



✅ Visualização (Insuficiência) concluída.


In [59]:
# =============================================================================
# Célula 9 (VERSÃO INSUFICIÊNCIA): Análise 6 - Cálculo de Métricas Preventivas
# =============================================================================

print("🚨 Iniciando Análise 6 (VERSÃO INSUFICIÊNCIA): Calculando KPIs Preventivos (Sinais de Alerta Pré-Insuficiência)...")

# 1. Criar um DataFrame focado APENAS em eventos ANTES do Dia 0
#    --- ESTA É A MUDANÇA ---
df_pre_insuficiencia = df_timeline_final_view_insuficiencia[
    df_timeline_final_view_insuficiencia['Dias_Desde_Evento_0'] < 0
].copy()

if df_pre_insuficiencia.empty:
    print("  - Nenhum evento registrado antes do 'Dia Zero' para os pacientes com Insuficiência.")

else:
    print(f"  - {df_pre_insuficiencia.shape[0]} eventos pré-insuficiência encontrados para análise.")


# --- 2. Calcular os KPIs Preventivos ---
# (Aplicando a lógica ao df_pre_insuficiencia)

# KPI 1: Qtd. Visitas Cardio Eletiva (Pré-Insuficiência)
df_cardio_pre = df_pre_insuficiencia[
    df_pre_insuficiencia['Evento_Label'] == 'VISITA: Cardio Eletiva'
]
kpi_cardio_pre = df_cardio_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Cardio_Eletiva_Pre_Insuficiencia')

# KPI 2: Qtd. Visitas Emergência (Pré-Insuficiência)
df_emerg_pre = df_pre_insuficiencia[
    df_pre_insuficiencia['Evento_Label'] == 'VISITA: Emergência'
]
kpi_emerg_pre = df_emerg_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Emergencia_Pre_Insuficiencia')

# KPI 3: Qtd. Procedimentos Chave (Pré-Insuficiência) - (Angio, Cateter, Stent)
df_proc_pre = df_pre_insuficiencia[
    df_pre_insuficiencia['Evento_Label'].str.contains('PROC:')
]
kpi_proc_pre = df_proc_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Procedimentos_Pre_Insuficiencia')

print("  - KPIs preventivos calculados.")

# --- 3. Consolidar a tabela de KPIs Preventivos ---

# Começamos com a lista completa dos 6 pacientes de Insuficiência
df_pacientes_insuficiencia = df_timeline_final_view_insuficiencia[['ID_PESSOA']].drop_duplicates()

# Juntar os KPIs preventivos
df_kpi_preventivo_insuficiencia = df_pacientes_insuficiencia.merge(kpi_cardio_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo_insuficiencia = df_kpi_preventivo_insuficiencia.merge(kpi_emerg_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo_insuficiencia = df_kpi_preventivo_insuficiencia.merge(kpi_proc_pre, on='ID_PESSOA', how='left')

# Substituir 'NaN' (Nulo) por 0. 'NaN' aqui significa "nenhum evento", ou seja, 0.
cols_kpi = ['Qtd_Cardio_Eletiva_Pre_Insuficiencia', 'Qtd_Emergencia_Pre_Insuficiencia', 'Qtd_Procedimentos_Pre_Insuficiencia']
df_kpi_preventivo_insuficiencia[cols_kpi] = df_kpi_preventivo_insuficiencia[cols_kpi].fillna(0).astype(int)

print("  - KPIs preventivos consolidados.")

# --- 4. Exibir Resultados ---
print("\n--- 🚨 Métricas Preventivas (Comportamento Pré-Insuficiência) ---")
print("(Contagem de eventos ANTES do 'Dia Zero' de cada paciente)")
display(df_kpi_preventivo_insuficiencia)

# --- 5. Calcular Médias Gerais ---
print("\n--- 📊 Resumo de Métricas Preventivas (Médias) ---")
media_cardio = df_kpi_preventivo_insuficiencia['Qtd_Cardio_Eletiva_Pre_Insuficiencia'].mean()
media_emerg = df_kpi_preventivo_insuficiencia['Qtd_Emergencia_Pre_Insuficiencia'].mean()
media_proc = df_kpi_preventivo_insuficiencia['Qtd_Procedimentos_Pre_Insuficiencia'].mean()

print(f"  - Média de Consultas Cardio Eletivas (Pré-Insuficiência): {media_cardio:.2f} por paciente")
print(f"  - Média de Visitas à Emergência (Pré-Insuficiência): {media_emerg:.2f} por paciente")
print(f"  - Média de Procedimentos-Chave (Pré-Insuficiência): {media_proc:.2f} por paciente")


print("\n✅ Análise 6 (KPIs Preventivos para Insuficiência) concluída.")

🚨 Iniciando Análise 6 (VERSÃO INSUFICIÊNCIA): Calculando KPIs Preventivos (Sinais de Alerta Pré-Insuficiência)...
  - 10 eventos pré-insuficiência encontrados para análise.
  - KPIs preventivos calculados.
  - KPIs preventivos consolidados.

--- 🚨 Métricas Preventivas (Comportamento Pré-Insuficiência) ---
(Contagem de eventos ANTES do 'Dia Zero' de cada paciente)


,ID_PESSOA,Qtd_Cardio_Eletiva_Pre_Insuficiencia,Qtd_Emergencia_Pre_Insuficiencia,Qtd_Procedimentos_Pre_Insuficiencia
0,133556,2,2,0
1,196323,2,0,0
2,204372,0,1,0
3,212498,0,2,0
4,357598,0,0,0
5,374899,1,0,0



--- 📊 Resumo de Métricas Preventivas (Médias) ---
  - Média de Consultas Cardio Eletivas (Pré-Insuficiência): 0.83 por paciente
  - Média de Visitas à Emergência (Pré-Insuficiência): 0.83 por paciente
  - Média de Procedimentos-Chave (Pré-Insuficiência): 0.00 por paciente

✅ Análise 6 (KPIs Preventivos para Insuficiência) concluída.


In [60]:
# =============================================================================
# Célula 4 (VERSÃO ANEURISMA): Jornada completa dos pacientes
# =============================================================================
import pandas as pd
import numpy as np

print("🔍 Iniciando análise: jornada completa dos pacientes com ANEURISMA...")

# 1) identificar IDs que em algum momento tiveram um dos CIDs (qualquer linha)
# --- NOVAS PALAVRAS-CHAVE ---
keywords_aneurisma = [
    'ANEURISMA E DISSECÇÃO DA AORTA',
    'ANEURISMA AORTICO DE LOCALIZAÇAO NAO ESPECIFICADA',
    'ANEURISMA DA AORTA ABDOMINAL, SEM MENÇAO DE',
    'ANEURISMA DE ARTERIA CORONARIA',
    'ANEURISMA DA AORTA EM DOENÇAS CLASSIFICADAS',
    'ANEURISMA DE LOCALIZAÇAO NAO ESPECIFICADA',
    'ANEURISMA CEREBRAL NAO-ROTO'
]
pattern_aneurisma = '|'.join(keywords_aneurisma)
print(f" ---------------------------------------------------------------------------------")
print(f"  - Padrão de busca: {pattern_aneurisma}")
print(f" ---------------------------------------------------------------------------------")


mask_aneurisma_any = df['DESCRICAO_CID'].str.contains(pattern_aneurisma, case=False, na=False)
ids_aneurisma = df.loc[mask_aneurisma_any, 'ID_PESSOA'].unique()
print(f"🫀 IDs únicos com registro de ANEURISMA: {len(ids_aneurisma)}")

# 2) obter todas as linhas desses pacientes (toda a jornada)
#    (Renomeado para df_jornada_aneurisma)
df_jornada_aneurisma = df[df['ID_PESSOA'].isin(ids_aneurisma)].copy()
print(f"📂 Registros desses pacientes (toda jornada): {df_jornada_aneurisma.shape[0]:,}")

# 3) garantir colunas de interesse como texto
for c in ['AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'MOMENTO', 'DESCRICAO_CID']:
    df_jornada_aneurisma[c] = df_jornada_aneurisma[c].astype(str)

# 4) separar antes/depois pelo campo MOMENTO
antes = df_jornada_aneurisma[df_jornada_aneurisma['MOMENTO'].str.upper().str.strip() == 'ANTES']
depois = df_jornada_aneurisma[df_jornada_aneurisma['MOMENTO'].str.upper().str.strip() == 'DEPOIS']
print(f"  - Registros 'Antes': {antes.shape[0]}. Registros 'Depois': {depois.shape[0]}.")

# 5) métricas desejadas:

# --- Métricas do período ANTES ---
mask_emerg_antes = antes['AGRUP_ASSISTENCIAL_G'].str.contains('EMERGENCIA', case=False, na=False)
mask_eletiva_antes = antes['AGRUP_ASSISTENCIAL_G'].str.contains('ELETIVA', case=False, na=False)
mask_cardio_antes = antes['ESPECIALIDADE'].str.contains('CARDIO', case=False, na=False)

emerg_antes_total = antes[mask_emerg_antes].shape[0]
emerg_antes_pacientes = antes[mask_emerg_antes]['ID_PESSOA'].nunique()
cardio_elet_antes_total = antes[mask_eletiva_antes & mask_cardio_antes].shape[0]
cardio_elet_antes_pacientes = antes[mask_eletiva_antes & mask_cardio_antes]['ID_PESSOA'].nunique()

# --- Métricas do período DEPOIS ---
mask_emerg_depois = depois['AGRUP_ASSISTENCIAL_G'].str.contains('EMERGENCIA', case=False, na=False)
mask_eletiva_depois = depois['AGRUP_ASSISTENCIAL_G'].str.contains('ELETIVA', case=False, na=False)
mask_cardio_depois = depois['ESPECIALIDADE'].str.contains('CARDIO', case=False, na=False)

emerg_depois_total = depois[mask_emerg_depois].shape[0]
emerg_depois_pacientes = depois[mask_emerg_depois]['ID_PESSOA'].nunique()
cardio_elet_depois_total = depois[mask_eletiva_depois & mask_cardio_depois].shape[0]
cardio_elet_depois_pacientes = depois[mask_eletiva_depois & mask_cardio_depois]['ID_PESSOA'].nunique()
print("  - Métricas 'Antes' e 'Depois' calculadas.")

# 6) montar resumo
resumo = pd.DataFrame({
    'Período': ['Antes', 'Depois'],
    'Pacientes únicos (nesse período)': [antes['ID_PESSOA'].nunique(), depois['ID_PESSOA'].nunique()],
    'Atendimentos emergência (total)': [emerg_antes_total, emerg_depois_total],
    'Pacientes com emergência': [emerg_antes_pacientes, emerg_depois_pacientes],
    'Consultas cardio eletiva (total)': [cardio_elet_antes_total, cardio_elet_depois_total],
    'Pacientes com cardio eletiva': [cardio_elet_antes_pacientes, cardio_elet_depois_pacientes]
})
print("\n📊 Resumo da Jornada (Aneurisma):")
display(resumo)

# 7) Top especialidades (antes / depois)
esp_antes = antes['ESPECIALIDADE'].value_counts().reset_index()
esp_antes.columns = ['Especialidade', 'Qtd_Antes']

esp_depois = depois['ESPECIALIDADE'].value_counts().reset_index()
esp_depois.columns = ['Especialidade', 'Qtd_Depois']

esp_comparativo = pd.merge(esp_antes, esp_depois, on='Especialidade', how='outer').fillna(0)
esp_comparativo = esp_comparativo.sort_values(by=['Qtd_Antes','Qtd_Depois'], ascending=False)
print("\n📋 Especialidades (antes/depois) — tabela completa (ordenada):")
display(esp_comparativo.head(50))

# 8) Linha do tempo por paciente: contagem de eventos por MOMENTO e tipo
def tipo_atendimento(row):
    a = str(row['AGRUP_ASSISTENCIAL_G']).upper()
    e = str(row['ESPECIALIDADE']).upper()
    if 'EMERGENCIA' in a: return 'EMERGENCIA'
    if 'ELETIVA' in a and 'CARDIO' in e: return 'CARDIO_ELETIVA'
    if 'ELETIVA' in a: return 'ELETIVA_OUTRA'
    return 'OUTRO'

df_jornada_aneurisma['TIPO_ATEND'] = df_jornada_aneurisma.apply(tipo_atendimento, axis=1)

timeline = df_jornada_aneurisma.groupby(['ID_PESSOA','MOMENTO','TIPO_ATEND']).size().unstack(fill_value=0).reset_index()
print("\n🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):")
display(timeline.head(50))

print("\n✅ Análise da jornada de ANEURISMA executada.")

🔍 Iniciando análise: jornada completa dos pacientes com ANEURISMA...
 ---------------------------------------------------------------------------------
  - Padrão de busca: ANEURISMA E DISSECÇÃO DA AORTA|ANEURISMA AORTICO DE LOCALIZAÇAO NAO ESPECIFICADA|ANEURISMA DA AORTA ABDOMINAL, SEM MENÇAO DE|ANEURISMA DE ARTERIA CORONARIA|ANEURISMA DA AORTA EM DOENÇAS CLASSIFICADAS|ANEURISMA DE LOCALIZAÇAO NAO ESPECIFICADA|ANEURISMA CEREBRAL NAO-ROTO
 ---------------------------------------------------------------------------------
🫀 IDs únicos com registro de ANEURISMA: 7
📂 Registros desses pacientes (toda jornada): 8,825
  - Registros 'Antes': 1197. Registros 'Depois': 7590.
  - Métricas 'Antes' e 'Depois' calculadas.

📊 Resumo da Jornada (Aneurisma):


,Período,Pacientes únicos (nesse período),Atendimentos emergência (total),Pacientes com emergência,Consultas cardio eletiva (total),Pacientes com cardio eletiva
0,Antes,7,5,2,18,7
1,Depois,7,11,4,15,7



📋 Especialidades (antes/depois) — tabela completa (ordenada):


,Especialidade,Qtd_Antes,Qtd_Depois
23,NAO INFORMADO,819.0,6057.0
9,CLINICA MEDICA,80.0,37.0
32,PATOLOGIA,46.0,29.0
2,CARDIOLOGIA,38.0,64.0
35,RADIOLOGIA,29.0,158.0
10,DERMATOLOGIA,25.0,9.0
14,FISIOTERAPIA,23.0,206.0
12,FARMACEUTICO,20.0,0.0
29,OFTALMOLOGIA,17.0,28.0
8,CIRURGIA VASCULAR,16.0,31.0



🧾 Timeline resumida por paciente (exemplo, primeiras 50 linhas):


TIPO_ATEND,ID_PESSOA,MOMENTO,CARDIO_ELETIVA,ELETIVA_OUTRA,EMERGENCIA,OUTRO
0,72119,ANTES,1,1,0,43
1,72119,DEPOIS,1,6,0,253
2,72119,MESMA DATA,0,0,0,2
3,225946,ANTES,2,5,0,84
4,225946,DEPOIS,1,11,5,6160
5,225946,MESMA DATA,0,0,0,11
6,269165,ANTES,1,14,0,172
7,269165,DEPOIS,6,10,2,569
8,269165,MESMA DATA,0,0,0,10
9,304264,ANTES,5,6,0,185



✅ Análise da jornada de ANEURISMA executada.


In [61]:
# =============================================================================
# Célula 5 (VERSÃO ANEURISMA): Análise 3 - Busca por procedimentos-chave
# =============================================================================
print("Iniciando Célula 5 (para Aneurisma): Busca por procedimentos-chave (Angio/Cateter/Stent)...")

# 1. Palavras-chave dos procedimentos (são as mesmas)
keywords_procedimentos = [
    'ANGIOTOMOGRAFIA CORONARIANA',
    'CATETERISMO CARDIACO',
    'IMPLANTE DE STENT CORONARIO'
]

# 2. Padrão regex com OU
#    (Esta variável 'pattern' será reutilizada na Célula 6)
pattern = '|'.join(keywords_procedimentos)
print(f"  - Padrão de busca por procedimentos: {pattern}")

# 3. Filtrar registros com os procedimentos
#    --- ESTA É A MUDANÇA ---
#    Agora procuramos no DataFrame df_jornada_aneurisma
try:
    df_jornada_aneurisma['SERVICO'] = df_jornada_aneurisma['SERVICO'].astype(str)
    mask = df_jornada_aneurisma['SERVICO'].str.contains(pattern, case=False, na=False)
    df_proc = df_jornada_aneurisma[mask].copy()
    print("  - Filtro de procedimentos aplicado ao df_jornada_aneurisma.")
except Exception as e:
    print(f"ERRO: O DataFrame 'df_jornada_aneurisma' não foi encontrado ou está sem a coluna 'SERVICO'. {e}")
    # Criar um DataFrame vazio para o código não quebrar
    df_proc = pd.DataFrame(columns=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'SERVICO', 'MOMENTO'])

# 4. REMOVER DUPLICATAS: mesmo paciente + mesma data + mesmo serviço = 1 evento
df_proc_unicos = df_proc.drop_duplicates(
    subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'SERVICO']
)

# 5. Selecionar apenas colunas relevantes
colunas_display = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'MOMENTO', 'SERVICO']

# 6. Resultados
if df_proc_unicos.empty:
    print("\nRESULTADO: Nenhum procedimento-chave encontrado para os pacientes com Aneurisma.")
else:
    print(f"\nSUCESSO! Encontrados {df_proc_unicos.shape[0]} procedimentos-chave únicos.\n")

    print("-- Detalhe dos Procedimentos (1 por data/serviço) --")
    display(df_proc_unicos[colunas_display]
            .sort_values(by=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO']))

    print("\n-- Resumo por Paciente e Momento --")
    resumo = (df_proc_unicos
                .groupby(['ID_PESSOA', 'MOMENTO', 'SERVICO'])
                .size()
                .reset_index(name='Qtd_Eventos'))
    display(resumo)

print("\nAnálise 3 (Célula 5) concluída para Aneurisma.")

Iniciando Célula 5 (para Aneurisma): Busca por procedimentos-chave (Angio/Cateter/Stent)...
  - Padrão de busca por procedimentos: ANGIOTOMOGRAFIA CORONARIANA|CATETERISMO CARDIACO|IMPLANTE DE STENT CORONARIO
  - Filtro de procedimentos aplicado ao df_jornada_aneurisma.

SUCESSO! Encontrados 9 procedimentos-chave únicos.

-- Detalhe dos Procedimentos (1 por data/serviço) --


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,MOMENTO,SERVICO
36728,72119,2024-05-16,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
36533,72119,2024-06-18,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
157251,225946,2023-07-26,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
189976,269165,2024-02-28,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
189462,269165,2024-08-26,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...
225380,304264,2023-08-30,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
307517,392016,2023-07-28,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
437558,677807,2025-04-28,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...
466355,762838,2024-09-21,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...



-- Resumo por Paciente e Momento --


,ID_PESSOA,MOMENTO,SERVICO,Qtd_Eventos
0,72119,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
1,72119,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
2,225946,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
3,269165,DEPOIS,CATETERISMO CARDIACO E E/OU D COM CINEANGIOCOR...,1
4,269165,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
5,304264,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
6,392016,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
7,677807,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1
8,762838,MESMA DATA,TC - ANGIOTOMOGRAFIA CORONARIANA (COM DIRETRIZ...,1



Análise 3 (Célula 5) concluída para Aneurisma.


In [62]:
# =============================================================================
# Célula 6 (VERSÃO ANEURISMA): Análise 4 - Cronologia Real (Timeline em Dias)
# =============================================================================
import numpy as np
import pandas as pd

print("⏳ Iniciando Análise 4 (VERSÃO ANEURISMA): Construindo a Linha do Tempo Real (em dias)...")

# 0. Definir o DataFrame de base
df_base_aneurisma = df_jornada_aneurisma.copy()

# 1. Garantir que as colunas de data e tipo de atendimento existem e estão corretas
try:
    df_base_aneurisma['DATA_ATENDIMENTO_FATO_PRO'] = pd.to_datetime(df_base_aneurisma['DATA_ATENDIMENTO_FATO_PRO'])

    # Garantir que colunas de texto são strings
    for col in ['SERVICO', 'AGRUP_ASSISTENCIAL_G', 'ESPECIALIDADE', 'DESCRICAO_CID']:
        if col in df_base_aneurisma.columns:
            df_base_aneurisma[col] = df_base_aneurisma[col].astype(str)

    print("  - Coluna de datas convertida para formato datetime.")
    print("  - Colunas de texto (incluindo SERVICO) garantidas como string.")

except Exception as e:
    print(f"Erro na preparação inicial: {e}.")

# (Opcional) Recriar TIPO_ATEND por segurança
if 'TIPO_ATEND' not in df_base_aneurisma.columns:
    print("  - Recriando coluna 'TIPO_ATEND'...")
    def tipo_atendimento(row):
        a = str(row['AGRUP_ASSISTENCIAL_G']).upper()
        e = str(row['ESPECIALIDADE']).upper()
        if 'EMERGENCIA' in a: return 'EMERGENCIA'
        if 'ELETIVA' in a and 'CARDIO' in e: return 'CARDIO_ELETIVA'
        if 'ELETIVA' in a: return 'ELETIVA_OUTRA'
        return 'OUTRO'
    df_base_aneurisma['TIPO_ATEND'] = df_base_aneurisma.apply(tipo_atendimento, axis=1)


# 2. Encontrar o "Dia Zero": a primeira data de registro de 'ANEURISMA' por paciente
#    (Verificar se 'pattern_aneurisma' existe da Célula 4)
try:
    pattern_aneurisma
except NameError:
    print("  - Recriando 'pattern_aneurisma' (keywords de CID)...")
    keywords_aneurisma = [
        'ANEURISMA E DISSECÇÃO DA AORTA',
        'ANEURISMA AORTICO DE LOCALIZAÇAO NAO ESPECIFICADA',
        'ANEURISMA DA AORTA ABDOMINAL, SEM MENÇAO DE',
        'ANEURISMA DE ARTERIA CORONARIA',
        'ANEURISMA DA AORTA EM DOENÇAS CLASSIFICADAS',
        'ANEURISMA DE LOCALIZAÇAO NAO ESPECIFICADA',
        'ANEURISMA CEREBRAL NAO-ROTO'
    ]
    pattern_aneurisma = '|'.join(keywords_aneurisma)

mask_aneurisma_temp = df_base_aneurisma['DESCRICAO_CID'].str.contains(pattern_aneurisma, case=False, na=False)
df_aneurisma_eventos = df_base_aneurisma[mask_aneurisma_temp]

event_0_dates = df_aneurisma_eventos.groupby('ID_PESSOA')['DATA_ATENDIMENTO_FATO_PRO'].min().reset_index()
event_0_dates.columns = ['ID_PESSOA', 'Data_Evento_0']

print("\n🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Aneurisma):")
display(event_0_dates)

# 3. Juntar a 'Data_Evento_0' a todos os registros dos pacientes
df_jornada_timeline_aneurisma = pd.merge(df_base_aneurisma, event_0_dates, on='ID_PESSOA', how='left')

# 4. Calcular os dias relativos ao "Dia Zero"
df_jornada_timeline_aneurisma['Dias_Desde_Evento_0'] = (df_jornada_timeline_aneurisma['DATA_ATENDIMENTO_FATO_PRO'] - df_jornada_timeline_aneurisma['Data_Evento_0']).dt.days

# 5. Definir as máscaras para TODOS os eventos-chave
#    (Verificar se 'pattern' de procedimentos existe da Célula 5)
try:
    pattern # Este é o 'pattern' dos procedimentos da Célula 5
except NameError:
    print("  - Recriando 'pattern' (keywords de procedimento)...")
    keywords_procedimentos = ['ANGIOTOMOGRAFIA CORONARIANA', 'CATETERISMO CARDIACO', 'IMPLANTE DE STENT CORONARIO']
    pattern = '|'.join(keywords_procedimentos)

print("  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline_aneurisma)...")

# Todas as máscaras são definidas no df_jornada_timeline_aneurisma
mask_proc = df_jornada_timeline_aneurisma['SERVICO'].str.contains(pattern, case=False, na=False)
mask_emerg = df_jornada_timeline_aneurisma['TIPO_ATEND'] == 'EMERGENCIA'
mask_cardio_elet = df_jornada_timeline_aneurisma['TIPO_ATEND'] == 'CARDIO_ELETIVA'
# Recriamos a máscara de aneurisma no DataFrame correto
mask_aneurisma = df_jornada_timeline_aneurisma['DESCRICAO_CID'].str.contains(pattern_aneurisma, case=False, na=False)


# 6. Filtrar o DataFrame para conter APENAS os eventos-chave
mask_key_events = mask_aneurisma | mask_proc | mask_emerg | mask_cardio_elet
df_timeline_filtrada = df_jornada_timeline_aneurisma[mask_key_events].copy()

# 7. Criar Etiquetas claras para os eventos (em ordem de prioridade)
conditions = [
    df_timeline_filtrada['SERVICO'].str.contains('IMPLANTE DE STENT', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('CATETERISMO CARDIACO', case=False, na=False),
    df_timeline_filtrada['SERVICO'].str.contains('ANGIOTOMOGRAFIA CORONARIANA', case=False, na=False),

    # --- A CORREÇÃO ESTÁ AQUI ---
    # Etiqueta "EVENTO" só é aplicada no "Dia Zero"
    (df_timeline_filtrada['DESCRICAO_CID'].str.contains(pattern_aneurisma, case=False, na=False)) &
    (df_timeline_filtrada['DATA_ATENDIMENTO_FATO_PRO'] == df_timeline_filtrada['Data_Evento_0']),
    # -----------------------------

    df_timeline_filtrada['TIPO_ATEND'] == 'EMERGENCIA',
    df_timeline_filtrada['TIPO_ATEND'] == 'CARDIO_ELETIVA'
]

choices = [
    'PROC: Implante de Stent',
    'PROC: Cateterismo Cardiaco',
    'PROC: Angiotomografia COR',
    'EVENTO: Registro Aneurisma', # Etiqueta atualizada
    'VISITA: Emergência',
    'VISITA: Cardio Eletiva'
]

df_timeline_filtrada['Evento_Label'] = np.select(conditions, choices, default='Outro (Reg. Aneurisma Sec.)')

# 8. Preparar e exibir a Tabela-Resumo Final
colunas_finais = ['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Dias_Desde_Evento_0', 'Evento_Label']
df_timeline_final_view_aneurisma = df_timeline_filtrada[colunas_finais]

# Remover os registros secundários (os 'Outros')
df_timeline_final_view_aneurisma = df_timeline_final_view_aneurisma[
    df_timeline_final_view_aneurisma['Evento_Label'].str.contains('Outro') == False
].copy()

# Ordenar e remover duplicatas
df_timeline_final_view_aneurisma = df_timeline_final_view_aneurisma.sort_values(by=['ID_PESSOA', 'Dias_Desde_Evento_0'])
df_timeline_final_view_aneurisma = df_timeline_final_view_aneurisma.drop_duplicates(
    subset=['ID_PESSOA', 'DATA_ATENDIMENTO_FATO_PRO', 'Evento_Label']
)

print("\n--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes com ANEURISMA) ---")
print("(Apenas o PRIMEIRO registro de Aneurisma é mostrado como 'EVENTO')")
display(df_timeline_final_view_aneurisma)

print("\n✅ Análise 4 (Célula 6) para ANEURISMA concluída. A cronologia está agora limpa.")

⏳ Iniciando Análise 4 (VERSÃO ANEURISMA): Construindo a Linha do Tempo Real (em dias)...
  - Coluna de datas convertida para formato datetime.
  - Colunas de texto (incluindo SERVICO) garantidas como string.

🗓️ Datas identificadas como 'Dia Zero' (Primeiro registro de Aneurisma):


,ID_PESSOA,Data_Evento_0
0,72119,2024-10-05
1,225946,2022-05-02
2,269165,2024-08-26
3,304264,2022-11-21
4,392016,2024-08-03
5,677807,2022-10-21
6,762838,2023-04-25


  - Definindo máscaras de filtro NO MESMO DataFrame (df_jornada_timeline_aneurisma)...

--- 🎬 LINHA DO TEMPO REAL (Filme dos Pacientes com ANEURISMA) ---
(Apenas o PRIMEIRO registro de Aneurisma é mostrado como 'EVENTO')


,ID_PESSOA,DATA_ATENDIMENTO_FATO_PRO,Dias_Desde_Evento_0,Evento_Label
97,72119,2024-05-07,-151,VISITA: Cardio Eletiva
269,72119,2024-05-16,-142,PROC: Angiotomografia COR
98,72119,2024-05-28,-130,VISITA: Cardio Eletiva
74,72119,2024-06-18,-109,PROC: Cateterismo Cardiaco
2,72119,2024-10-05,0,EVENTO: Registro Aneurisma
...,...,...,...,...
8482,762838,2023-04-25,0,EVENTO: Registro Aneurisma
8623,762838,2024-06-24,426,VISITA: Cardio Eletiva
8624,762838,2024-09-19,513,VISITA: Cardio Eletiva
8781,762838,2024-09-21,515,PROC: Angiotomografia COR



✅ Análise 4 (Célula 6) para ANEURISMA concluída. A cronologia está agora limpa.


In [63]:
# =============================================================================
# Célula 7 (VERSÃO ANEURISMA): Visualização da Cronologia (Timeline Plot)
# =============================================================================
import plotly.express as px

print("📊 Iniciando Célula 7 (VERSÃO ANEURISMA): Geração do Gráfico de Cronologia...")

# --- Preparação para o Gráfico ---
# 1. Usar o DataFrame final do ANEURISMA
df_plot = df_timeline_final_view_aneurisma.copy()

# 2. Garantir que o ID do Paciente é tratado como texto (categoria)
df_plot['ID_PESSOA'] = df_plot['ID_PESSOA'].astype(str)

# --- Calcular o Range Completo do Eixo X ---
# Vamos encontrar o dia mínimo (ex: -151) e o máximo
min_dias = df_plot['Dias_Desde_Evento_0'].min()
max_dias = df_plot['Dias_Desde_Evento_0'].max()

# Adicionar uma "margem" visual (ex: 30 dias)
min_dias_com_margem = min_dias - 30
max_dias_com_margem = max_dias + 30

print(f"  - Dados preparados. Range do eixo X definido de {min_dias_com_margem} até {max_dias_com_margem} dias.")

# --- Criação do Gráfico Interativo ---
fig = px.scatter(
    data_frame=df_plot,
    x='Dias_Desde_Evento_0',
    y='ID_PESSOA',
    color='Evento_Label',                  # Define as cores com base na etiqueta do evento
    hover_data=[                         # O que mostrar ao passar o rato
        'DATA_ATENDIMENTO_FATO_PRO',
        'Evento_Label',
        'Dias_Desde_Evento_0'
    ],
    labels={                             # Renomear os eixos para ficarem claros
        'Dias_Desde_Evento_0': 'Dias desde o Evento Zero (1º Aneurisma)',
        'ID_PESSOA': 'Paciente',
        'Evento_Label': 'Tipo de Evento'
    },
    # Título atualizado
    title='Cronologia da Jornada do Paciente (Dias desde o 1º Aneurisma)'
)

# --- Melhorias no Gráfico ---
# 1. Adicionar uma linha vertical vermelha no "Dia 0"
fig.add_vline(
    x=0,
    line_width=2,
    line_dash="dash",
    line_color="red",
    annotation_text="Dia 0 (Aneurisma)", # Texto atualizado
    annotation_position="bottom right"
)

# 2. Melhorar a visualização para 7 pacientes
#    (Etiquetas ativadas no eixo Y, pois são poucos pacientes)
fig.update_yaxes(
    type='category',
    showticklabels=True,
    title='Pacientes'
)

# 3. Ordenar o eixo Y (pacientes) E APLICAR O NOVO RANGE DO EIXO X
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_range=[min_dias_com_margem, max_dias_com_margem],
    height=500 # Altura de 500px é boa para 7 pacientes
)

print("  - Gráfico gerado com sucesso.")

# --- Exibir o Gráfico ---
fig.show()

print("\n✅ Visualização (Aneurisma) concluída.")

📊 Iniciando Célula 7 (VERSÃO ANEURISMA): Geração do Gráfico de Cronologia...
  - Dados preparados. Range do eixo X definido de -861 até 1200 dias.
  - Gráfico gerado com sucesso.



✅ Visualização (Aneurisma) concluída.


In [64]:
# =============================================================================
# Célula 9 (VERSÃO ANEURISMA): Análise 6 - Cálculo de Métricas Preventivas
# =============================================================================

print("🚨 Iniciando Análise 6 (VERSÃO ANEURISMA): Calculando KPIs Preventivos (Sinais de Alerta Pré-Aneurisma)...")

# 1. Criar um DataFrame focado APENAS em eventos ANTES do Dia 0
#    --- ESTA É A MUDANÇA ---
df_pre_aneurisma = df_timeline_final_view_aneurisma[
    df_timeline_final_view_aneurisma['Dias_Desde_Evento_0'] < 0
].copy()

if df_pre_aneurisma.empty:
    print("  - Nenhum evento registrado antes do 'Dia Zero' para os pacientes com Aneurisma.")

else:
    print(f"  - {df_pre_aneurisma.shape[0]} eventos pré-aneurisma encontrados para análise.")


# --- 2. Calcular os KPIs Preventivos ---
# (Aplicando a lógica ao df_pre_aneurisma)

# KPI 1: Qtd. Visitas Cardio Eletiva (Pré-Aneurisma)
df_cardio_pre = df_pre_aneurisma[
    df_pre_aneurisma['Evento_Label'] == 'VISITA: Cardio Eletiva'
]
kpi_cardio_pre = df_cardio_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Cardio_Eletiva_Pre_Aneurisma')

# KPI 2: Qtd. Visitas Emergência (Pré-Aneurisma)
df_emerg_pre = df_pre_aneurisma[
    df_pre_aneurisma['Evento_Label'] == 'VISITA: Emergência'
]
kpi_emerg_pre = df_emerg_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Emergencia_Pre_Aneurisma')

# KPI 3: Qtd. Procedimentos Chave (Pré-Aneurisma) - (Angio, Cateter, Stent)
df_proc_pre = df_pre_aneurisma[
    df_pre_aneurisma['Evento_Label'].str.contains('PROC:')
]
kpi_proc_pre = df_proc_pre.groupby('ID_PESSOA').size().reset_index(name='Qtd_Procedimentos_Pre_Aneurisma')

print("  - KPIs preventivos calculados.")

# --- 3. Consolidar a tabela de KPIs Preventivos ---

# Começamos com a lista completa dos 7 pacientes de Aneurisma
df_pacientes_aneurisma = df_timeline_final_view_aneurisma[['ID_PESSOA']].drop_duplicates()

# Juntar os KPIs preventivos
df_kpi_preventivo_aneurisma = df_pacientes_aneurisma.merge(kpi_cardio_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo_aneurisma = df_kpi_preventivo_aneurisma.merge(kpi_emerg_pre, on='ID_PESSOA', how='left')
df_kpi_preventivo_aneurisma = df_kpi_preventivo_aneurisma.merge(kpi_proc_pre, on='ID_PESSOA', how='left')

# Substituir 'NaN' (Nulo) por 0. 'NaN' aqui significa "nenhum evento", ou seja, 0.
cols_kpi = ['Qtd_Cardio_Eletiva_Pre_Aneurisma', 'Qtd_Emergencia_Pre_Aneurisma', 'Qtd_Procedimentos_Pre_Aneurisma']
df_kpi_preventivo_aneurisma[cols_kpi] = df_kpi_preventivo_aneurisma[cols_kpi].fillna(0).astype(int)

print("  - KPIs preventivos consolidados.")

# --- 4. Exibir Resultados ---
print("\n--- 🚨 Métricas Preventivas (Comportamento Pré-Aneurisma) ---")
print("(Contagem de eventos ANTES do 'Dia Zero' de cada paciente)")
display(df_kpi_preventivo_aneurisma)

# --- 5. Calcular Médias Gerais ---
print("\n--- 📊 Resumo de Métricas Preventivas (Médias) ---")
media_cardio = df_kpi_preventivo_aneurisma['Qtd_Cardio_Eletiva_Pre_Aneurisma'].mean()
media_emerg = df_kpi_preventivo_aneurisma['Qtd_Emergencia_Pre_Aneurisma'].mean()
media_proc = df_kpi_preventivo_aneurisma['Qtd_Procedimentos_Pre_Aneurisma'].mean()

print(f"  - Média de Consultas Cardio Eletivas (Pré-Aneurisma): {media_cardio:.2f} por paciente")
print(f"  - Média de Visitas à Emergência (Pré-Aneurisma): {media_emerg:.2f} por paciente")
print(f"  - Média de Procedimentos-Chave (Pré-Aneurisma): {media_proc:.2f} por paciente")


print("\n✅ Análise 6 (KPIs Preventivos para Aneurisma) concluída.")

🚨 Iniciando Análise 6 (VERSÃO ANEURISMA): Calculando KPIs Preventivos (Sinais de Alerta Pré-Aneurisma)...
  - 21 eventos pré-aneurisma encontrados para análise.
  - KPIs preventivos calculados.
  - KPIs preventivos consolidados.

--- 🚨 Métricas Preventivas (Comportamento Pré-Aneurisma) ---
(Contagem de eventos ANTES do 'Dia Zero' de cada paciente)


,ID_PESSOA,Qtd_Cardio_Eletiva_Pre_Aneurisma,Qtd_Emergencia_Pre_Aneurisma,Qtd_Procedimentos_Pre_Aneurisma
0,72119,2,0,2
1,225946,0,0,0
2,269165,3,0,1
3,304264,3,0,0
4,392016,2,2,1
5,677807,0,1,0
6,762838,4,0,0



--- 📊 Resumo de Métricas Preventivas (Médias) ---
  - Média de Consultas Cardio Eletivas (Pré-Aneurisma): 2.00 por paciente
  - Média de Visitas à Emergência (Pré-Aneurisma): 0.43 por paciente
  - Média de Procedimentos-Chave (Pré-Aneurisma): 0.57 por paciente

✅ Análise 6 (KPIs Preventivos para Aneurisma) concluída.


In [65]:
# =============================================================================
# Célula 10: Análise 7 - Tabela Comparativa Final de KPIs Preventivos
# =============================================================================
import pandas as pd
import numpy as np

print("🚀 Iniciando Análise 7: Criando a Tabela Comparativa Final de Sinais de Alerta...")

# 1. Definir uma função auxiliar (baseada na nossa Célula 9)
#    Esta função recebe um DataFrame de cronologia e um nome,
#    e devolve as médias preventivas
def calcular_medias_preventivas(df_timeline, condition_name):
    print(f"  - Processando dados para: {condition_name}...")

    try:
        # 1. Filtrar eventos pré-evento
        df_pre = df_timeline[df_timeline['Dias_Desde_Evento_0'] < 0].copy()

        # 2. Criar lista completa de pacientes
        df_pacientes = df_timeline[['ID_PESSOA']].drop_duplicates()

        # Se não houver eventos pré, as médias são 0
        if df_pre.empty:
            return {
                'Condição': condition_name,
                'Nº Pacientes': len(df_pacientes),
                'Média Consultas Cardio Eletivas': 0.0,
                'Média Visitas à Emergência': 0.0,
                'Média Procedimentos-Chave': 0.0
            }

        # 3. Calcular KPIs individuais
        kpi_cardio_pre = df_pre[df_pre['Evento_Label'] == 'VISITA: Cardio Eletiva'].groupby('ID_PESSOA').size().reset_index(name='Qtd_Cardio_Eletiva')
        kpi_emerg_pre = df_pre[df_pre['Evento_Label'] == 'VISITA: Emergência'].groupby('ID_PESSOA').size().reset_index(name='Qtd_Emergencia')
        kpi_proc_pre = df_pre[df_pre['Evento_Label'].str.contains('PROC:')].groupby('ID_PESSOA').size().reset_index(name='Qtd_Procedimentos')

        # 4. Consolidar tabela por paciente
        df_kpi_preventivo = df_pacientes.merge(kpi_cardio_pre, on='ID_PESSOA', how='left')
        df_kpi_preventivo = df_kpi_preventivo.merge(kpi_emerg_pre, on='ID_PESSOA', how='left')
        df_kpi_preventivo = df_kpi_preventivo.merge(kpi_proc_pre, on='ID_PESSOA', how='left')

        cols_kpi = ['Qtd_Cardio_Eletiva', 'Qtd_Emergencia', 'Qtd_Procedimentos']
        df_kpi_preventivo[cols_kpi] = df_kpi_preventivo[cols_kpi].fillna(0).astype(int)

        # 5. Calcular médias finais
        media_cardio = df_kpi_preventivo['Qtd_Cardio_Eletiva'].mean()
        media_emerg = df_kpi_preventivo['Qtd_Emergencia'].mean()
        media_proc = df_kpi_preventivo['Qtd_Procedimentos'].mean()

        return {
            'Condição': condition_name,
            'Nº Pacientes': len(df_pacientes),
            'Média Consultas Cardio Eletivas': media_cardio,
            'Média Visitas à Emergência': media_emerg,
            'Média Procedimentos-Chave': media_proc
        }

    except Exception as e:
        print(f"  - ERRO ao processar {condition_name}: {e}")
        return None

# 2. Lista para guardar os resultados
kpi_summary_list = []

# 3. Lista dos DataFrames que criámos
#    (Usamos try-except para o caso de alguma análise não ter sido executada)
analises_a_fazer = [
    ('Infarto', 'df_timeline_final_view'),
    ('Angina', 'df_timeline_final_view_angina'),
    ('Arritmia', 'df_timeline_final_view_arritmia'),
    ('Insuficiência', 'df_timeline_final_view_insuficiencia'),
    ('Aneurisma', 'df_timeline_final_view_aneurisma')
]

for nome, df_nome in analises_a_fazer:
    try:
        # Pega o DataFrame real (ex: df_timeline_final_view_angina)
        df_real = globals()[df_nome]
        resultado = calcular_medias_preventivas(df_real, nome)
        if resultado:
            kpi_summary_list.append(resultado)
    except KeyError:
        print(f"  - AVISO: DataFrame '{df_nome}' para '{nome}' não foi encontrado. A pular.")

# 4. Criar o DataFrame final
if not kpi_summary_list:
    print("\nERRO: Nenhum DataFrame de cronologia foi encontrado. Não é possível gerar o resumo.")
else:
    df_comparativo_final = pd.DataFrame(kpi_summary_list)
    df_comparativo_final = df_comparativo_final.set_index('Condição')

    print("\n\n" + "="*80)
    print("--- 🚀 TABELA COMPARATIVA FINAL: PADRÕES DE RISCO PREVENTIVO (MÉDIAS PRÉ-'DIA ZERO') ---")
    print("="*80)

    # Exibir a tabela final arredondada
    display(df_comparativo_final.round(2))

print("\n✅ Análise comparativa concluída. Todo o pipeline foi finalizado com sucesso!")

🚀 Iniciando Análise 7: Criando a Tabela Comparativa Final de Sinais de Alerta...
  - Processando dados para: Infarto...
  - Processando dados para: Angina...
  - Processando dados para: Arritmia...
  - Processando dados para: Insuficiência...
  - Processando dados para: Aneurisma...


--- 🚀 TABELA COMPARATIVA FINAL: PADRÕES DE RISCO PREVENTIVO (MÉDIAS PRÉ-'DIA ZERO') ---


,Nº Pacientes,Média Consultas Cardio Eletivas,Média Visitas à Emergência,Média Procedimentos-Chave
Condição,,,,
Infarto,2,1.00,0.00,0.00
Angina,64,1.80,1.09,0.67
Arritmia,7,1.14,4.14,0.14
Insuficiência,6,0.83,0.83,0.00
Aneurisma,7,2.00,0.43,0.57



✅ Análise comparativa concluída. Todo o pipeline foi finalizado com sucesso!


In [66]:
# =============================================================================
# Célula 11: Análise 8 - Tabela Comparativa Final por JANELA DE TEMPO
# =============================================================================
import pandas as pd
import numpy as np

print("🚀 Iniciando Análise 8: Criando a Tabela Comparativa Final por Janela de Tempo...")

# 1. Definir a função auxiliar (agora com limite de dias)
def calcular_medias_preventivas_janela(df_timeline, condition_name, window_name, dias_limite=None):

    # Mensagem de log para sabermos o que está a acontecer
    log_msg = f"  - Processando: {condition_name} ({window_name})..."
    print(log_msg, end=" ")

    try:
        # 1. Filtrar eventos pré-evento DENTRO da janela de tempo
        if dias_limite is None:
            # Período Completo (Tudo < 0)
            df_pre = df_timeline[df_timeline['Dias_Desde_Evento_0'] < 0].copy()
        else:
            # Janela específica (ex: >= -365 e < 0)
            df_pre = df_timeline[
                (df_timeline['Dias_Desde_Evento_0'] < 0) &
                (df_timeline['Dias_Desde_Evento_0'] >= -dias_limite)
            ].copy()

        # 2. Criar lista completa de pacientes
        df_pacientes = df_timeline[['ID_PESSOA']].drop_duplicates()

        # Se não houver eventos pré, as médias são 0
        if df_pre.empty:
            print("Nenhum evento pré-janela.")
            return {
                'Condição': condition_name,
                'Janela de Tempo': window_name,
                'Nº Pacientes': len(df_pacientes),
                'Média Consultas Cardio Eletivas': 0.0,
                'Média Visitas à Emergência': 0.0,
                'Média Procedimentos-Chave': 0.0
            }

        # 3. Calcular KPIs individuais
        kpi_cardio_pre = df_pre[df_pre['Evento_Label'] == 'VISITA: Cardio Eletiva'].groupby('ID_PESSOA').size().reset_index(name='Qtd_Cardio_Eletiva')
        kpi_emerg_pre = df_pre[df_pre['Evento_Label'] == 'VISITA: Emergência'].groupby('ID_PESSOA').size().reset_index(name='Qtd_Emergencia')
        kpi_proc_pre = df_pre[df_pre['Evento_Label'].str.contains('PROC:')].groupby('ID_PESSOA').size().reset_index(name='Qtd_Procedimentos')

        # 4. Consolidar tabela por paciente
        df_kpi_preventivo = df_pacientes.merge(kpi_cardio_pre, on='ID_PESSOA', how='left')
        df_kpi_preventivo = df_kpi_preventivo.merge(kpi_emerg_pre, on='ID_PESSOA', how='left')
        df_kpi_preventivo = df_kpi_preventivo.merge(kpi_proc_pre, on='ID_PESSOA', how='left')

        cols_kpi = ['Qtd_Cardio_Eletiva', 'Qtd_Emergencia', 'Qtd_Procedimentos']
        df_kpi_preventivo[cols_kpi] = df_kpi_preventivo[cols_kpi].fillna(0).astype(int)

        # 5. Calcular médias finais
        media_cardio = df_kpi_preventivo['Qtd_Cardio_Eletiva'].mean()
        media_emerg = df_kpi_preventivo['Qtd_Emergencia'].mean()
        media_proc = df_kpi_preventivo['Qtd_Procedimentos'].mean()

        print("OK.")

        return {
            'Condição': condition_name,
            'Janela de Tempo': window_name,
            'Nº Pacientes': len(df_pacientes),
            'Média Consultas Cardio Eletivas': media_cardio,
            'Média Visitas à Emergência': media_emerg,
            'Média Procedimentos-Chave': media_proc
        }

    except Exception as e:
        print(f"ERRO: {e}")
        return None

# 2. Lista para guardar os resultados
kpi_summary_list = []

# 3. Lista dos DataFrames que criámos
analises_a_fazer = [
    ('Infarto', 'df_timeline_final_view'),
    ('Angina', 'df_timeline_final_view_angina'),
    ('Arritmia', 'df_timeline_final_view_arritmia'),
    ('Insuficiência', 'df_timeline_final_view_insuficiencia'),
    ('Aneurisma', 'df_timeline_final_view_aneurisma')
]

# 4. Lista das Janelas de Tempo
time_windows = [
    ('Último 1 Ano', 365),
    ('Últimos 2 Anos', 730),
    ('Período Completo', None) # None = sem limite
]

# 5. Loop Aninhado: 5 Doenças x 3 Janelas de Tempo
for nome, df_nome in analises_a_fazer:
    try:
        df_real = globals()[df_nome]

        for window_name, dias in time_windows:
            resultado = calcular_medias_preventivas_janela(df_real, nome, window_name, dias)
            if resultado:
                kpi_summary_list.append(resultado)

    except KeyError:
        print(f"  - AVISO: DataFrame '{df_nome}' para '{nome}' não foi encontrado. A pular.")

# 6. Criar o DataFrame final
if not kpi_summary_list:
    print("\nERRO: Nenhum DataFrame de cronologia foi encontrado. Não é possível gerar o resumo.")
else:
    df_comparativo_final_janelas = pd.DataFrame(kpi_summary_list)
    df_comparativo_final_janelas = df_comparativo_final_janelas.set_index(['Condição', 'Janela de Tempo'])

    print("\n\n" + "="*80)
    print("--- 🚀 TABELA COMPARATIVA FINAL: PADRÕES DE RISCO (1 Ano, 2 Anos, Completo) ---")
    print("="*80)

    # Exibir a tabela final arredondada e ordenada
    display(df_comparativo_final_janelas.sort_index().round(2))

print("\n✅ Análise comparativa por JANELA DE TEMPO concluída. Este é o fim do pipeline!")

🚀 Iniciando Análise 8: Criando a Tabela Comparativa Final por Janela de Tempo...
  - Processando: Infarto (Último 1 Ano)... OK.
  - Processando: Infarto (Últimos 2 Anos)... OK.
  - Processando: Infarto (Período Completo)... OK.
  - Processando: Angina (Último 1 Ano)... OK.
  - Processando: Angina (Últimos 2 Anos)... OK.
  - Processando: Angina (Período Completo)... OK.
  - Processando: Arritmia (Último 1 Ano)... OK.
  - Processando: Arritmia (Últimos 2 Anos)... OK.
  - Processando: Arritmia (Período Completo)... OK.
  - Processando: Insuficiência (Último 1 Ano)... OK.
  - Processando: Insuficiência (Últimos 2 Anos)... OK.
  - Processando: Insuficiência (Período Completo)... OK.
  - Processando: Aneurisma (Último 1 Ano)... OK.
  - Processando: Aneurisma (Últimos 2 Anos)... OK.
  - Processando: Aneurisma (Período Completo)... OK.


--- 🚀 TABELA COMPARATIVA FINAL: PADRÕES DE RISCO (1 Ano, 2 Anos, Completo) ---


Nº Pacientes  Média Consultas Cardio Eletivas  \
Condição      Janela de Tempo                                                   
Aneurisma     Período Completo             7                             2.00   
              Último 1 Ano                 7                             1.86   
              Últimos 2 Anos               7                             2.00   
Angina        Período Completo            64                             1.80   
              Último 1 Ano                64                             1.23   
              Últimos 2 Anos              64                             1.66   
Arritmia      Período Completo             7                             1.14   
              Último 1 Ano                 7                             0.86   
              Últimos 2 Anos               7                             1.14   
Infarto       Período Completo             2                             1.00   
              Último 1 Ano                 2                             1.00   
              Últimos 2 Anos               2                             1.00   
Insuficiência Período Completo             6                             0.83   
              Último 1 Ano                 6                             0.83   
              Últimos 2 Anos               6                             0.83   

                                Média Visitas à Emergência  \
Condição      Janela de Tempo                                
Aneurisma     Período Completo                        0.43   
              Último 1 Ano                            0.14   
              Últimos 2 Anos                          0.14   
Angina        Período Completo                        1.09   
              Último 1 Ano                            0.45   
              Últimos 2 Anos                          0.86   
Arritmia      Período Completo                        4.14   
              Último 1 Ano                            2.00   
              Últimos 2 Anos                          3.29   
Infarto       Período Completo                        0.00   
              Último 1 Ano                            0.00   
              Últimos 2 Anos                          0.00   
Insuficiência Período Completo                        0.83   
              Último 1 Ano                            0.67   
              Últimos 2 Anos                          0.83   

                                Média Procedimentos-Chave  
Condição      Janela de Tempo                              
Aneurisma     Período Completo                       0.57  
              Último 1 Ano                           0.43  
              Últimos 2 Anos                         0.57  
Angina        Período Completo                       0.67  
              Último 1 Ano                           0.64  
              Últimos 2 Anos                         0.67  
Arritmia      Período Completo                       0.14  
              Último 1 Ano                           0.14  
              Últimos 2 Anos                         0.14  
Infarto       Período Completo                       0.00  
              Último 1 Ano                           0.00  
              Últimos 2 Anos                         0.00  
Insuficiência Período Completo                       0.00  
              Último 1 Ano                           0.00  
              Últimos 2 Anos                         0.00


✅ Análise comparativa por JANELA DE TEMPO concluída. Este é o fim do pipeline!
